In [1]:
"""Fruity Monster — Arcade 3.3.3, Python functions (no custom game classes).
Smooth-graphics + smooth-motion edition  +  LEVEL SELECT (5 levels)  +  MAPS.

Keep this file, common PNGs, best.pt and hand_landmarker.task in ONE directory.
   Store character poses in purple/, green/, orange/, pink/ folders (or Characters/).

HD RESPONSIVE EDITION: 1920x1080 display-ready image preparation;
16:9-friendly expanded camera view without cropping, borders, or distortion.
The physics still uses 900x600 logical coordinates to keep original gameplay.
Original low-resolution PNGs cannot gain real detail just by scaling.
Jupyter: %run Fruity_Monster.py

Eat or Don't Eat: MediaPipe hand gestures for movement and jump; mouth tracked with YOLO.
Intro UI: pixel START -> character selection -> 3-mode menu (all three modes playable).
EAT OR DON'T EAT now opens a LEVEL SELECT page (5 levels, stars, locks).
Say the Fruit: needs SpeechRecognition + microphone, pyttsx3 for the spoken
word and internet for Google speech recognition.

NEW in this version  ->  search for "VOICE2" to find every change:
  * VOICE2: SAY THE FRUIT now uses the NIGHT pixel background.
  * VOICE2: step 1 = press LISTEN and hear the word. Only after the word has
    been spoken does the microphone unlock (step 2 = say the word).
  * VOICE2: after a wrong / unclear answer a RED "SKIP" button appears
    (it replaces the old NEXT button). It is hidden again once you get it right.
  * VOICE2: top bar (same style as the game HUD) shows all 10 fruits. Every
    fruit you say correctly lights up and fills its bar segment. BACK moved
    into the top bar too.

NEW  ->  search for "FOREST", "MEADOW", "PLATFORMS" or "BOSS":
  * FOREST : level 3 has its own map (teal sky, pixel sun, pine trees, rocks).
  * MEADOW : level 5 (BOSS) has its own map (pale sky, pipes, mushrooms, keys).
  * PLATFORMS: floating platforms with fruit on top (jump up and eat from
    above). Level 3 = a few, level 4 = more, level 5 = most + 2-storey ones.
  * BOSS: BOSS LEVEL intro, red BOSS HUD, golden KING FRUITS worth +3,
    "BOSS DEFEATED!" on the win screen, crown on the level card.
"""
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import json
import math
import sys
import threading
import time
import arcade
import random
import re
import cv2
import numpy as np
from PIL import Image, ImageOps
from ultralytics import YOLO
import speech_recognition as sr
import pyttsx3

ROOT = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
# Existing gameplay uses a stable 900x600 coordinate system. Changing this
# directly would alter pits, collision ranges, and character movement.
# Instead, render those coordinates sharply into a responsive 16:9-capable view.
W, H = 900, 600
DESIGN_OUTPUT_W, DESIGN_OUTPUT_H = 1920, 1080  # target DISPLAY resolution, not physics
ASSET_QUALITY_SCALE = 2.2  # source PNG textures pre-rendered for large windows
GROUND, WORLD_W = 98, 7000          # longer level so items can be further apart
SPEED, JUMP_SPEED, GRAVITY = 200, 400, 600   # SPEED was 260 (slower walk)
ITEM_GAP = (260, 340)               # min / max distance between items (was ~150)
HOLES = [(1100, 1310), (2160, 2370), (3370, 3580), (4370, 4580)]
WHITE = (255, 255, 255)
INK = (68, 36, 105)

# ---- feel / quality knobs (tweak freely) ----------------------------------
ART_FACES_RIGHT = True      # set False if your monster PNGs face LEFT
ACCEL, FRICTION = 2600.0, 3200.0
COYOTE, JUMP_BUFFER = 0.10, 0.12
VARIABLE_JUMP = False       # True = tapping the key makes a shorter hop
WIN_HOP_SPEED = 3.0         # victory hop speed (was 6 = too fast). Lower = slower
AIR_SPEED_MULT = 2.0        # horizontal speed in the air (x SPEED) -> wider arc
AIR_FRICTION = 0.15         # 1 = stops instantly in air, small = keeps gliding
CAM_SMOOTH = 8.0            # higher = camera follows tighter
PARALLAX = 0.12             # background scroll speed relative to the world
YOLO_IMGSZ, YOLO_CONF = 320, 0.25   # smaller = faster reaction
YOLO_DEBOUNCE = 1           # 1 = react instantly (2-3 = steadier but delayed)
YOLO_LOST_FRAMES = 1        # MOUTH LOST: frames with NO mouth found before it counts as CLOSED
# MediaPipe Hand Landmarker: one camera shared with mouth YOLO.
# Tilt a hand left/right to walk; point it UP to jump.
HAND_MODEL_PATH = ROOT / 'hand_landmarker.task'
HAND_STALE_SECONDS = 0.45

# Say the Fruit layout (VOICE2): one pixel card on the right holds everything.
VP_X, VP_L, VP_B, VP_W, VP_H = 640, 460, 118, 360, 392   # card centre-x / left / bottom / size
VOICE_LISTEN_Y = 282                                      # LISTEN button row
MIC_Y, MIC_R = 190, 36                                    # mic row
MIC_X_SOLO, MIC_X_SKIP = 640, 550                         # mic centred, or shifted when SKIP shows
SKIP_X, SKIP_W, SKIP_H = 702, 120, 48
VOICE_NEXT_DELAY = 3.0      # VOICE2: seconds before the next fruit after a correct answer

# Add later fruits here after creating matching PNG files (e.g. banana.png).
FRUIT_TYPES = ['apple', 'banana', 'strawberry', 'orange', 'grapes',
               'watermelon', 'mango', 'kiwi', 'pineapple', 'cherries']

FRUIT_NAMES = {
    'apple': 'Apple',
    'banana': 'Banana',
    'strawberry': 'Strawberry',
    'orange': 'Orange',
    'grapes': 'Grapes',
    'watermelon': 'Watermelon',
    'mango': 'Mango',
    'kiwi': 'Kiwi',
    'pineapple': 'Pineapple',
    'cherries': 'Cherries'
}

BAD_ITEMS = ['sock', 'shoe', 'toy', 'pencil', 'toothbrush', 'soap', 'eraser']

# ---- LEVELS: edit this table to change difficulty -------------------------
# fruit  : the level's main fruit (shown on the card)
# pool   : fruits that can appear in the level
# goal   : fruits to eat to win
# bad_n  : how many bad-item TYPES appear (first N of BAD_ITEMS)
# bad    : how many bad items are placed in the level
# color  : card frame colour
# map    : 'night', 'sunset', 'volcano', 'forest' or 'meadow' (see MAP_THEMES)
# plat   : PLATFORMS - chance (0..1) that an item sits on a floating platform
# high   : PLATFORMS - chance that such a platform is a 2-storey one
# boss   : BOSS - True = boss intro, red HUD, boss win text, crown on the card
# king   : BOSS - how many KING FRUITS (+3) are hidden on the highest platforms
LEVELS = [
    dict(fruit='apple', pool=['apple'], goal=5, bad_n=2, bad=2,
         color=(235, 70, 70), map='night'),
    dict(fruit='banana', pool=['banana', 'apple'], goal=6, bad_n=3, bad=4,
         color=(250, 215, 70), map='night'),
    dict(fruit='strawberry', pool=['strawberry', 'banana', 'apple'], goal=8,
         bad_n=4, bad=5, color=(230, 90, 140), map='forest', plat=0.35, high=0.0),
    dict(fruit='orange', pool=['orange', 'strawberry', 'banana', 'apple'],
         goal=10, bad_n=6, bad=6, color=(250, 150, 50), map='sunset',
         plat=0.50, high=0.30),
    dict(fruit='watermelon', pool=['watermelon', 'watermelon'] + FRUIT_TYPES,
         goal=10, bad_n=7, bad=7, color=(255, 210, 95), map='meadow',   # BOSS
         plat=0.65, high=0.50, boss=True, king=2),
]
NUM_LEVELS = len(LEVELS)
PROGRESS_FILE = ROOT / 'progress.json'

BURST_COLORS = {
    'apple': (235, 70, 70), 'banana': (250, 215, 70),
    'strawberry': (230, 60, 100), 'orange': (250, 150, 50),
    'grapes': (150, 90, 200),
    'watermelon': (245, 80, 105),
    'mango': (255, 185, 60),
    'kiwi': (135, 210, 70),
    'pineapple': (245, 200, 65),
    'cherries': (215, 60, 90),
}
CONFETTI = [(255, 120, 150), (255, 213, 90), (110, 200, 255),
            (140, 230, 140), (190, 140, 255)]

try:
    window = arcade.Window(W, H, 'Fruity Monster | Arcade', update_rate=1/60,
                           draw_rate=1/60, vsync=True,
                           antialiasing=True, samples=4, resizable=True)
except Exception as err:  # some GPUs/drivers refuse MSAA
    print('MSAA unavailable, falling back:', err)
    window = arcade.Window(W, H, 'Fruity Monster | Arcade', update_rate=1/60,
                           draw_rate=1/60, vsync=True, resizable=True)

try:
    PIXEL = float(window.get_pixel_ratio())
except Exception:
    PIXEL = 1.0

# Responsive EXPAND viewport: keeps aspect ratios, avoids bars AND avoids
# cropping the existing game buttons. Extra width/height shows more scenery.
view = {'scale': 1.0, 'world_left': 0.0, 'world_bottom': 0.0,
        'world_right': float(W), 'world_top': float(H)}
ui_camera = arcade.Camera2D(
    position=(0, 0),
    projection=arcade.LRBT(0, W, 0, H),
    viewport=arcade.LBWH(0, 0, W, H),
    window=window,
)


def configure_viewport():
    win_w, win_h = window.get_size()
    win_w, win_h = max(1, win_w), max(1, win_h)
    # EXPAND rather than COVER: no crop, no black bars, no stretching.
    # Example: maximized 1920x1080 shows the original 900x600 canvas
    # plus 83.33 world units of extra scenery at each side.
    scale = min(win_w / W, win_h / H)
    world_w, world_h = win_w / scale, win_h / scale
    left, bottom = (W - world_w) / 2, (H - world_h) / 2
    view.update(scale=scale, world_left=left, world_bottom=bottom,
                world_right=left+world_w, world_top=bottom+world_h)
    ui_camera.viewport = arcade.LBWH(0, 0, win_w, win_h)
    ui_camera.projection = arcade.LRBT(left, left + world_w,
                                       bottom, bottom + world_h)


def screen_to_game(x, y):
    # Click coordinates must undergo exactly the same EXPAND transform as drawing.
    configure_viewport()
    return (view['world_left'] + x / view['scale'],
            view['world_bottom'] + y / view['scale'])


def on_resize(width, height):
    configure_viewport()


# ---------------------------------------------------------------------------
# Textures: pre-scale with LANCZOS to the size they're drawn at (x pixel ratio).
# ---------------------------------------------------------------------------
LANCZOS = getattr(Image, 'Resampling', Image).LANCZOS
_raw = {}


# The 4 character folders can sit directly beside this script (as in the
# user's screenshot), OR inside a single Characters/ folder.
CHARACTER_NAMES = ('purple', 'green', 'orange', 'pink')
CHARACTERS_DIR = (ROOT / 'Characters' if (ROOT / 'Characters').is_dir() else ROOT)

# ---------------------------------------------------------------------------
# CHAR_MOTION: how each character MOVES (visual only, physics unchanged).
#   hover      : pixels the sprite floats above the ground (pink = flying)
#   float_amp  : size of the up/down floating wobble while hovering
#   step_hop   : extra hop (pixels) on every step while walking
#   waddle     : sideways rocking angle (degrees) on every step
#   squash     : squash & stretch on each step (0.05 = 5 %)
#   anim       : walk-frame speed multiplier (pink flaps faster)
# Works with ANY pose, so it still looks like walking when the mouth is OPEN.
# ---------------------------------------------------------------------------
CHAR_MOTION = {
    'purple': dict(hover=0,  float_amp=0, step_hop=0,  waddle=0, squash=0.00, anim=1.0),
    'green':  dict(hover=0,  float_amp=0, step_hop=9,  waddle=5, squash=0.04, anim=1.0),
    'orange': dict(hover=0,  float_amp=0, step_hop=9,  waddle=6, squash=0.05, anim=1.0),
    'pink':   dict(hover=30, float_amp=5, step_hop=4,  waddle=3, squash=0.02, anim=1.6),
}


def char_motion():
    return CHAR_MOTION.get(g['selected_character'], CHAR_MOTION['purple'])


def raw_image(name, character=None):
    if character is None:
        path = ROOT / f'{name}.png'  # fruits, bad items, hearts, background
        key = ('common', name)
    else:
        path = CHARACTERS_DIR / character / f'{name}.png'
        # The original purple artwork can lack game_over.png.
        # Show its sad pose in that case, without affecting other characters.
        if character == 'purple' and name == 'game_over' and not path.is_file():
            path = CHARACTERS_DIR / character / 'sad.png'
        key = (character, name)
    if key not in _raw:
        if not path.is_file():
            raise FileNotFoundError(
                f'Missing image: {path}\n'
                'Put the purple, green, orange and pink folders beside this Python file '
                'or inside a Characters folder. Each needs the standard sprite PNG names.')
        _raw[key] = Image.open(path).convert('RGBA')
    return _raw[key]


def make_tex(name, w, h, flip=False, character=None):
    img = raw_image(name, character=character)
    if flip:
        img = ImageOps.mirror(img)
    # Prepared for 1080p (and higher-DPI) output; Arcade then shrinks to
    # window size rather than stretching a 900px-wide raster on maximize.
    # Upscaling a tiny source PNG cannot recreate details not in the source.
    target_w = max(1, round(w * PIXEL * ASSET_QUALITY_SCALE))
    target_h = max(1, round(h * PIXEL * ASSET_QUALITY_SCALE))
    # Cap oversized rasters to be friendlier to older laptop GPUs.
    cap = min(1.0, 3072 / max(target_w, target_h))
    size = (max(1, round(target_w * cap)), max(1, round(target_h * cap)))
    return arcade.Texture(img.resize(size, LANCZOS))


MONSTER_POSES = ('idle', 'mouth_open', 'walk_1', 'walk_2', 'walk_3', 'walk_4',
                 'happy', 'sad', 'win', 'game_over')
MONSTER_GAME = (175, 182)
MONSTER_MENU = (290, 303)
ITEM_SIZE, BIG_ITEM_SIZE, HEART_SIZE = 59, 83, 40

tex = {}  # Character keys: (character, pose, size, flipped)
for character in CHARACTER_NAMES:
    for pose in MONSTER_POSES:
        tex[(character, pose, 'game', False)] = make_tex(
            pose, *MONSTER_GAME, character=character)
        tex[(character, pose, 'game', True)] = make_tex(
            pose, *MONSTER_GAME, flip=True, character=character)
        tex[(character, pose, 'menu', False)] = make_tex(
            pose, *MONSTER_MENU, character=character)
for name in ('heart_full', 'heart_empty'):
    tex[(name, 'hud', False)] = make_tex(name, HEART_SIZE, HEART_SIZE)
for name in FRUIT_TYPES:
    tex[(name, 'item', False)] = make_tex(name, ITEM_SIZE, ITEM_SIZE)
    tex[(name, 'big', False)] = make_tex(name, BIG_ITEM_SIZE, BIG_ITEM_SIZE)
for name in BAD_ITEMS:
    tex[(name, 'item', False)] = make_tex(name, ITEM_SIZE, ITEM_SIZE)

tex[('bg', 'bg', False)] = make_tex('pastel_scenery', W, H)
tex[('bg', 'bg', True)] = make_tex('pastel_scenery', W, H, flip=True)

# One shared dictionary instead of defining any custom class.
g = dict(screen='start', selected_character='purple', x=155.0, y=GROUND, vy=0.0, vx=0.0, cam=0.0, dcam=0,
         facing=1, keys=set(), hearts=3, score=0, objects=[],
         level=0, unlocked=1, stars=[0] * NUM_LEVELS, goal=10,   # LEVELS
         mouth='OFF', cam_status='off', cam_stop=None, cam_thread=None,
         hand_gesture='STOP', hand_updated=0.0, hand_status='WAITING',
         hand_jump_latched=False, hand_jump_direction=0,
         eat_cooldown=0.0, animation_time=0.0, t=0.0,
         coyote=0.0, jump_buffer=0.0, land=0.0, walk_phase=0.0,
         shake=0.0, particles=[], mouse=(-1, -1),
         invulnerable=0.0, banner='', banner_time=0.0,
         show_debug=False, heart_lost_t=0.0, heart_lost_i=0, fruit_pop=0.0,  # HUD
         voice_fruit='apple', voice_result='Press LISTEN to hear the word.',
         voice_busy=False, voice_future=None, voice_score=0,
         voice_message_time=0.0, voice_index=0,
         voice_wait=0.0, voice_finished=False,
         voice_order=[], voice_reaction='idle', voice_task_kind='speak',
         voice_recording=False, voice_stop_event=None,
         voice_heard=False, voice_wrong=False,                         # VOICE2
         voice_got=set(), voice_last='', voice_pop=0.0,                # VOICE2
         platforms=[], boss_intro=0.0,                                 # PLATFORMS / BOSS
         boss_hurt=0.0, boss_roar=0.0, boss_throw=1.2, boss_hazards=[])
voice_executor = ThreadPoolExecutor(max_workers=1)


def current_character():
    return g['selected_character']


def character_tex(pose, size='game', flip=False, character=None):
    return tex[(character or current_character(), pose, size, flip)]


# ---------------------------------------------------------------------------
# LEVELS: progress is saved to progress.json beside this file.
# ---------------------------------------------------------------------------
def load_progress():
    try:
        data = json.loads(PROGRESS_FILE.read_text(encoding='utf-8'))
        stars = [max(0, min(3, int(s))) for s in data.get('stars', [])]
        stars = (stars + [0] * NUM_LEVELS)[:NUM_LEVELS]
        g['stars'] = stars
        g['unlocked'] = max(1, min(NUM_LEVELS, int(data.get('unlocked', 1))))
    except Exception:
        pass  # first run / unreadable file -> keep defaults


def save_progress():
    try:
        PROGRESS_FILE.write_text(
            json.dumps({'unlocked': g['unlocked'], 'stars': g['stars']}),
            encoding='utf-8')
    except Exception as err:
        print('Could not save progress:', err)


load_progress()

# Original game sounds: place the WAV files beside this Python file.
SOUND_FILES = {
    'eat': 'eat.wav', 'wrong': 'wrong.wav', 'heart': 'heart.wav',
    'win': 'win.wav', 'game_over': 'game_over.wav',
    'happy': 'happy.wav', 'sad': 'sad.wav', 'jump': 'jump.wav',
    'music': 'background_music.wav'
}
sounds = {}
for sound_name, filename in SOUND_FILES.items():
    sound_path = ROOT / filename
    if sound_path.is_file():
        try:
            sounds[sound_name] = arcade.load_sound(str(sound_path))
        except Exception as err:
            print('Sound unavailable:', filename, err)
    else:
        print('Missing sound:', filename)

music_player = None


def play_effect(name, volume=0.55):
    sound = sounds.get(name)
    if sound is not None:
        try:
            arcade.play_sound(sound, volume=volume)
        except Exception as err:
            print('Sound playback:', name, err)


def stop_music():
    global music_player
    if music_player is not None:
        try:
            arcade.stop_sound(music_player)
        except Exception:
            pass
        music_player = None


def start_music():
    global music_player
    if music_player is None and 'music' in sounds:
        try:
            music_player = arcade.play_sound(sounds['music'], volume=0.12, loop=True)
        except Exception as err:
            print('Background music:', err)


# ---------------------------------------------------------------------------
# Small helpers
# ---------------------------------------------------------------------------
_text_cache = {}


def txt(msg, x, y, color=WHITE, size=16, anchor_x='left', anchor_y='baseline',
        bold=False):
    """Cached replacement for arcade.draw_text (much faster, same look)."""
    key = (msg, size, color, anchor_x, anchor_y, bold)
    t = _text_cache.get(key)
    if t is None:
        if len(_text_cache) > 400:
            _text_cache.clear()
        t = arcade.Text(msg, x, y, color, size, anchor_x=anchor_x,
                        anchor_y=anchor_y, bold=bold)
        _text_cache[key] = t
    t.x, t.y = x, y
    t.draw()


def blit(texture, cx, cy, w, h, alpha=255, angle=0.0):
    arcade.draw_texture_rect(texture, arcade.XYWH(cx, cy, w, h),
                             angle=angle, alpha=alpha)


def approach(v, target, step):
    if v < target:
        return min(v + step, target)
    return max(v - step, target)


def hit(x, y, cx, cy, width, height):
    return abs(x-cx) <= width/2 and abs(y-cy) <= height/2


def fancy_button(cx, cy, width, height, label, active=True):
    # Rounded capsule with a soft bottom shadow, hover lift and a gloss line.
    hover = active and hit(g['mouse'][0], g['mouse'][1], cx, cy, width, height)
    if active:
        base = (180, 116, 232) if hover else (155, 88, 210)
        dark = (87, 42, 129)
    else:
        base, dark = (139, 128, 157), (102, 91, 121)
    cy = cy + (3 if hover else 0)
    r = height / 2
    arcade.draw_lrbt_rectangle_filled(cx-width/2+r, cx+width/2-r,
                                     cy-height/2-5, cy+height/2-5, dark)
    arcade.draw_circle_filled(cx-width/2+r, cy-5, r, dark)
    arcade.draw_circle_filled(cx+width/2-r, cy-5, r, dark)
    arcade.draw_lrbt_rectangle_filled(cx-width/2+r, cx+width/2-r,
                                     cy-height/2, cy+height/2, base)
    arcade.draw_circle_filled(cx-width/2+r, cy, r, base)
    arcade.draw_circle_filled(cx+width/2-r, cy, r, base)
    arcade.draw_lrbt_rectangle_filled(cx-width/2+r, cx+width/2-r,
                                     cy+height*0.14, cy+height/2-4,
                                     (255, 255, 255, 38))
    txt(label, cx, cy+1, WHITE, 16, anchor_x='center', anchor_y='center',
        bold=True)


# ---------------------------------------------------------------------------
# Particles
# ---------------------------------------------------------------------------
def spawn_burst(x, y, color, n=10, speed=260, size=(3, 7), grav=900.0,
                life=(0.35, 0.7)):
    for _ in range(n):
        a = random.uniform(0, math.tau)
        s = random.uniform(0.3, 1.0) * speed
        lf = random.uniform(*life)
        g['particles'].append({
            'x': x, 'y': y, 'vx': math.cos(a)*s, 'vy': math.sin(a)*s + 60,
            'life': lf, 'max': lf, 'color': color,
            'size': random.uniform(*size), 'grav': grav})
    if len(g['particles']) > 320:
        del g['particles'][:len(g['particles']) - 320]


def spawn_dust(x, y, n=4):
    for _ in range(n):
        lf = random.uniform(0.25, 0.45)
        g['particles'].append({
            'x': x + random.uniform(-14, 14), 'y': y,
            'vx': random.uniform(-70, 70), 'vy': random.uniform(25, 90),
            'life': lf, 'max': lf, 'color': (255, 255, 255),
            'size': random.uniform(2.5, 5.0), 'grav': 120.0})


def update_particles(dt):
    ps = g['particles']
    for p in ps:
        p['life'] -= dt
        p['vy'] -= p['grav'] * dt
        p['x'] += p['vx'] * dt
        p['y'] += p['vy'] * dt
    g['particles'] = [p for p in ps if p['life'] > 0]


def draw_particles():
    cam = g['dcam']
    for p in g['particles']:
        k = p['life'] / p['max']
        r, gr, b = p['color']
        arcade.draw_circle_filled(p['x']-cam, p['y'], p['size']*(0.4+0.6*k),
                                  (r, gr, b, int(255*k)))


# ---------------------------------------------------------------------------
# World drawing
# ---------------------------------------------------------------------------
def draw_background(scroll=0.0):
    # Mirrored repeating background covers the entire *visible* projection,
    # including the extra scenery exposed by wide monitors.
    # (Still used by the main menu; SAY THE FRUIT now uses the night scene.)
    shift = scroll * PARALLAX
    left, right = view['world_left'], view['world_right']
    first = math.floor((left + shift) / W) - 1
    last = math.ceil((right + shift) / W) + 1
    for idx in range(first, last + 1):
        x = idx * W - shift
        if x > right or x + W < left:
            continue
        texture = tex[('bg', 'bg', bool(idx % 2))]
        blit(texture, x + W / 2, H / 2, W + 1, H)
    # For very tall window ratios, extend the edge sky/ground tint behind scene.


def ground_exists(x):
    return not any(a <= x <= b for a, b in HOLES)


# PLATFORMS: one-way floating platforms. p = {'x': centre, 'y': TOP surface, 'w': width}
PLAT_PAD = 14      # the feet may still stand this far past a platform's edge
PLAT_THICK = 26    # drawing only


def floor_under(x, y_prev):
    """Highest surface at/below height y_prev under x (ground or a platform).
    Platforms are one-way: you jump up through them and land on top."""
    # PIT FIX: ground only catches you if your feet were still at ground level.
    # Once you have sunk into a pit you can no longer "pop" back up at its edge.
    best = GROUND if (ground_exists(x) and y_prev >= GROUND - 4) else None
    for p in g['platforms']:
        if (p['y'] <= y_prev + 4 and (best is None or p['y'] > best)
                and abs(x - p['x']) <= p['w'] / 2 + PLAT_PAD):
            best = p['y']
    return best


def standing_on():
    """Height of the surface the monster stands on, or None while airborne."""
    fl = floor_under(g['x'], g['y'])
    return fl if (fl is not None and g['y'] <= fl + 2) else None


SEGMENTS = []
_prev = 0
for _a, _b in HOLES:
    SEGMENTS.append((_prev, _a))
    _prev = _b
SEGMENTS.append((_prev, WORLD_W))

SOIL, SOIL_DARK = (177, 133, 100), (150, 110, 82)
GRASS, GRASS_LIGHT, GRASS_TUFT = (96, 184, 78), (126, 212, 98), (82, 163, 66)


def pixel_rect(left, bottom, width, height, color):
    arcade.draw_lrbt_rectangle_filled(left, left + width, bottom, bottom + height, color)


# ---------------------------------------------------------------------------
# MAPS: each level uses a theme (see 'map' in LEVELS). Drawn with code only.
# ---------------------------------------------------------------------------
MAP_THEMES = {
    'night': dict(
        kind='night', sky=[(24, 31, 78)], text=(255, 240, 255),
        pit=(10, 12, 35, 235),
        soil=(147, 77, 74), soil_dark=(119, 62, 80), soil_dot=(175, 101, 75),
        grass=(97, 175, 78), grass_light=(150, 209, 88), tuft=(69, 139, 94),
        star=(255, 216, 97), cloud=(127, 89, 188), cloud_hi=(231, 140, 192),
        bush=(69, 139, 94), bush_hi=(94, 177, 104)),
    'sunset': dict(
        kind='sunset',
        sky=[(92, 60, 140), (150, 80, 160), (214, 100, 150),
             (250, 140, 120), (255, 185, 110), (255, 222, 150)],
        text=(255, 255, 255), pit=(60, 25, 70, 235),
        soil=(176, 104, 92), soil_dark=(140, 76, 84), soil_dot=(205, 134, 108),
        grass=(116, 176, 84), grass_light=(170, 216, 100), tuft=(88, 150, 70),
        cloud=(255, 196, 190), cloud_hi=(255, 150, 170),
        mt_far=(168, 92, 150), mt_near=(120, 64, 128)),
    'volcano': dict(
        kind='volcano',
        sky=[(35, 12, 40), (70, 16, 50), (110, 24, 50),
             (160, 40, 40), (210, 80, 40)],
        text=(255, 238, 220), pit=(255, 110, 40, 240),
        soil=(78, 38, 48), soil_dark=(52, 24, 38), soil_dot=(120, 60, 60),
        grass=(105, 62, 58), grass_light=(255, 140, 50), tuft=(80, 46, 46),
        cloud=(95, 52, 70), cloud_hi=(140, 76, 84),
        mt_far=(72, 28, 50), mt_near=(46, 20, 38), cap=(255, 125, 50)),
    # FOREST (level 3): teal sky, pixel sun, pine trees, rocks.
    'forest': dict(
        kind='forest',
        sky=[(60, 110, 100), (76, 128, 116), (94, 148, 136),
             (116, 168, 156), (142, 190, 180), (172, 210, 204)],
        text=(255, 255, 255), pit=(22, 38, 46, 235),
        soil=(122, 80, 52), soil_dark=(92, 58, 42), soil_dot=(150, 102, 66),
        grass=(88, 168, 56), grass_light=(142, 208, 84), tuft=(62, 138, 52),
        cloud=(244, 250, 255), cloud_hi=(196, 222, 244),
        tree=(78, 168, 58), tree_dark=(42, 120, 52), tree_light=(150, 224, 130),
        trunk=(116, 66, 50),
        rock=(112, 110, 122), rock_hi=(158, 156, 168), rock_dark=(80, 78, 92)),
    # MEADOW (level 5 / BOSS): pale sky, round trees, red pipes, mushrooms, keys.
    'meadow': dict(
        kind='meadow',
        sky=[(254, 253, 238), (254, 250, 222), (253, 243, 196),
             (251, 235, 170), (248, 226, 142)],
        text=(52, 40, 110), pit=(34, 26, 60, 235),
        soil=(166, 82, 42), soil_dark=(120, 52, 40), soil_dot=(198, 112, 56),
        grass=(92, 182, 48), grass_light=(146, 222, 64), tuft=(58, 140, 40),
        cloud=(206, 228, 248), cloud_hi=(238, 247, 254),
        tree=(92, 182, 56), tree_dark=(56, 134, 48), tree_light=(150, 224, 100),
        trunk=(128, 78, 52)),
}
NIGHT_STARS = ((80, 550), (225, 372), (835, 545), (790, 322), (128, 275),
               (625, 405), (430, 520), (520, 300), (330, 470), (700, 480))


def current_theme():
    lvl = max(0, min(g['level'], NUM_LEVELS - 1))
    return MAP_THEMES[LEVELS[lvl].get('map', 'night')]


def _tiles(scroll, factor, period, drift=0.0):
    """Yield screen-x of each repeating tile of a parallax layer."""
    shift = scroll * factor + drift
    first = math.floor((view['world_left'] + shift) / period) - 1
    last = math.ceil((view['world_right'] + shift) / period) + 1
    for i in range(first, last + 1):
        yield i * period - shift


def pixel_cloud(cx, cy, body, hi):
    for dx, dy, ww, hh in ((-50, 0, 102, 22), (-27, 18, 75, 24),
                           (-6, 36, 45, 18)):
        pixel_rect(cx + dx, cy + dy, ww, hh, body)
    pixel_rect(cx - 38, cy + 8, 76, 11, hi)


def pixel_mountain(cx, base, w, h, color, cap=None, steps=6):
    sh = h / steps
    for k in range(steps):
        sw = w * (1 - k / steps)
        c = cap if (cap and k == steps - 1) else color
        pixel_rect(cx - sw / 2, base + k * sh, sw, sh + 1, c)


MEADOW_SPARKS = ((60, 480), (300, 540), (620, 500), (820, 420), (470, 455))


def mix(a, b, t):
    return tuple(int(a[i] + (b[i] - a[i]) * t) for i in range(3))


def pixel_sun(cx, cy, r, body, rim, ray, step=8):
    """FOREST: blocky pixel sun built from rows, with square rays around it."""
    rows = int(r // step)
    for j in range(-rows, rows):
        yy = j * step
        half = math.sqrt(max(0.0, r * r - (yy + step / 2) ** 2))
        half = round(half / step) * step
        if half <= 0:
            continue
        pixel_rect(cx - half, cy + yy, half * 2, step, rim)
        if half > step * 1.5:
            pixel_rect(cx - half + step, cy + yy, half * 2 - 2 * step, step, body)
    for k in range(12):
        a = k * math.tau / 12
        d = r + (step * 2.2 if k % 2 == 0 else step * 1.4)
        px = round((cx + math.cos(a) * d) / step) * step
        py = round((cy + math.sin(a) * d) / step) * step
        size = step * (2 if k % 2 == 0 else 1.3)
        pixel_rect(px - size / 2, py - size / 2, size, size, ray)


def pixel_pine(cx, base, h, th, far=False):
    """FOREST: stacked pixel pine tree (far=True fades it into the sky)."""
    body, dark, light, trunk = th['tree'], th['tree_dark'], th['tree_light'], th['trunk']
    if far:
        haze = th['sky'][3]
        body, dark, light, trunk = (mix(c, haze, 0.5)
                                    for c in (body, dark, light, trunk))
    trunk_h = h * 0.16
    pixel_rect(cx - h * 0.035, base, h * 0.07, trunk_h, trunk)
    tiers = 5
    tier_h = h * 0.84 / tiers
    for k in range(tiers):
        w = h * 0.52 * (1 - k * 0.17)
        y0 = base + trunk_h + k * tier_h * 0.9
        pixel_rect(cx - w / 2, y0, w, tier_h, body)
        pixel_rect(cx - w / 2, y0, w * 0.2, tier_h, light)
        pixel_rect(cx + w * 0.3, y0, w * 0.2, tier_h, dark)
        pixel_rect(cx - w / 2, y0, w, tier_h * 0.16, dark)
    pixel_rect(cx - h * 0.03, base + trunk_h + tiers * tier_h * 0.9 + tier_h * 0.1,
               h * 0.06, h * 0.04, light)


def pixel_rock(cx, base, w, h, th):
    """FOREST: little grey pixel rock."""
    pixel_rect(cx - w / 2, base, w, h * 0.45, th['rock'])
    pixel_rect(cx - w * 0.38, base + h * 0.45, w * 0.76, h * 0.30, th['rock'])
    pixel_rect(cx - w * 0.22, base + h * 0.75, w * 0.44, h * 0.25, th['rock'])
    pixel_rect(cx - w / 2, base, w * 0.18, h * 0.45, th['rock_hi'])
    pixel_rect(cx - w * 0.38, base + h * 0.45, w * 0.14, h * 0.30, th['rock_hi'])
    pixel_rect(cx + w * 0.2, base, w * 0.3, h * 0.45, th['rock_dark'])


def pixel_round_tree(cx, base, r, th, far=False):
    """MEADOW: round pixel tree (canopy built from rows)."""
    body, dark, light, trunk = th['tree'], th['tree_dark'], th['tree_light'], th['trunk']
    if far:
        haze = th['sky'][2]
        body, dark, light, trunk = (mix(c, haze, 0.35)
                                    for c in (body, dark, light, trunk))
    trunk_h = r * 0.9
    pixel_rect(cx - r * 0.14, base, r * 0.28, trunk_h, trunk)
    cy = base + trunk_h + r * 0.8
    step = 6
    for j in range(-int(r // step), int(r // step)):
        yy = j * step
        half = math.sqrt(max(0.0, r * r - (yy + step / 2) ** 2))
        half = round(half / step) * step
        if half <= 0:
            continue
        pixel_rect(cx - half, cy + yy, half * 2, step, body)
        pixel_rect(cx + half * 0.35, cy + yy, half * 0.65, step, dark)
        pixel_rect(cx - half * 0.7, cy + yy, half * 0.18, step, light)


def pixel_pipe(cx, base, h, th=None):
    """MEADOW: red pixel pipe."""
    edge, body, hi, shade = (70, 36, 98), (206, 64, 78), (238, 132, 136), (132, 44, 96)
    pixel_rect(cx - 27, base, 54, h - 20, edge)
    pixel_rect(cx - 24, base, 48, h - 20, body)
    pixel_rect(cx - 20, base, 9, h - 20, hi)
    pixel_rect(cx + 11, base, 13, h - 20, shade)
    pixel_rect(cx - 35, base + h - 24, 70, 24, edge)
    pixel_rect(cx - 32, base + h - 22, 64, 20, body)
    pixel_rect(cx - 28, base + h - 22, 11, 20, hi)
    pixel_rect(cx + 16, base + h - 22, 16, 20, shade)


def pixel_mushroom(cx, base, s=1.0):
    """MEADOW: red spotted mushroom."""
    pixel_rect(cx - 6 * s, base, 12 * s, 3 * s, (160, 158, 172))
    pixel_rect(cx - 5 * s, base + 3 * s, 10 * s, 8 * s, (232, 230, 238))
    pixel_rect(cx - 15 * s, base + 11 * s, 30 * s, 8 * s, (214, 58, 66))
    pixel_rect(cx - 11 * s, base + 19 * s, 22 * s, 5 * s, (214, 58, 66))
    pixel_rect(cx - 15 * s, base + 11 * s, 30 * s, 2 * s, (150, 40, 80))
    pixel_rect(cx - 9 * s, base + 14 * s, 5 * s, 4 * s, WHITE)
    pixel_rect(cx + 4 * s, base + 14 * s, 5 * s, 4 * s, WHITE)
    pixel_rect(cx - 2 * s, base + 20 * s, 4 * s, 3 * s, WHITE)


def pixel_key(cx, cy):
    """MEADOW: floating golden key (decoration only)."""
    gold, dark = (248, 214, 84), (206, 160, 50)
    pixel_rect(cx - 14, cy + 3, 12, 3, gold)
    pixel_rect(cx - 14, cy - 6, 12, 3, gold)
    pixel_rect(cx - 14, cy - 6, 3, 12, gold)
    pixel_rect(cx - 5, cy - 6, 3, 12, dark)
    pixel_rect(cx - 2, cy - 1, 24, 3, gold)
    pixel_rect(cx + 14, cy - 5, 3, 5, gold)
    pixel_rect(cx + 20, cy - 4, 3, 4, dark)


def pixel_crown(cx, base, w, h):
    """BOSS: little golden crown (king fruit, boss card)."""
    gold, dark, red = (255, 205, 70), (214, 150, 40), (235, 75, 100)
    pw = w / 5
    pixel_rect(cx - w / 2, base, w, h * 0.4, gold)
    pixel_rect(cx - w / 2, base, w, h * 0.12, dark)
    pixel_rect(cx - w / 2, base + h * 0.4, pw, h * 0.45, gold)
    pixel_rect(cx - pw / 2, base + h * 0.4, pw, h * 0.60, gold)
    pixel_rect(cx + w / 2 - pw, base + h * 0.4, pw, h * 0.45, gold)
    pixel_rect(cx - pw / 2, base + h * 0.14, pw, h * 0.2, red)


def draw_map_background(scroll=0.0):
    th = current_theme()
    kind = th['kind']
    L, R = view['world_left'], view['world_right']
    B, T = view['world_bottom'], view['world_top']
    # Sky (horizontal colour bands; first/last band extend past the edges).
    bands = th['sky']
    n = len(bands)
    bh = H / n
    for i, col in enumerate(bands):
        top = T if i == 0 else H - i * bh
        bot = B if i == n - 1 else H - (i + 1) * bh
        arcade.draw_lrbt_rectangle_filled(L, R, bot, top, col)

    if kind == 'night':
        for ox in _tiles(scroll, 0.05, 900):
            for i, (sx, sy) in enumerate(NIGHT_STARS):
                x = ox + sx
                if x < L - 20 or x > R + 20:
                    continue
                tw = 0.5 + 0.5 * math.sin(g['t'] * 2.7 + i * 1.5)
                arcade.draw_circle_filled(x, sy, 18, (255, 223, 106, int(45 * tw)))
                pixel_rect(x - 3, sy - 13, 6, 26, th['star'])
                pixel_rect(x - 13, sy - 3, 26, 6, th['star'])
    elif kind == 'sunset':
        for ox in _tiles(scroll, 0.03, 1800):
            arcade.draw_circle_filled(ox + 1000, 360, 105, (255, 230, 160, 60))
            arcade.draw_circle_filled(ox + 1000, 360, 70, (255, 240, 170))
        for ox in _tiles(scroll, 0.15, 900):
            for mx, w, h in ((120, 380, 190), (520, 420, 230), (820, 340, 170)):
                pixel_mountain(ox + mx, GROUND - 4, w, h, th['mt_far'])
    elif kind == 'volcano':
        for ox in _tiles(scroll, 0.15, 900):
            for mx, w, h in ((150, 400, 230), (560, 460, 280), (850, 340, 190)):
                pixel_mountain(ox + mx, GROUND - 4, w, h, th['mt_far'],
                               cap=th['cap'])
    elif kind == 'forest':
        for ox in _tiles(scroll, 0.03, 1800):          # FOREST: pixel sun
            pixel_sun(ox + 330, 455, 62, (244, 204, 72), (252, 232, 120),
                      (250, 226, 110))
        for ox in _tiles(scroll, 0.15, 600):           # FOREST: far hazy pines
            for tx, h in ((70, 250), (300, 200), (470, 270)):
                pixel_pine(ox + tx, GROUND - 4, h, th, far=True)
    elif kind == 'meadow':
        for ox in _tiles(scroll, 0.04, 900):           # MEADOW: yellow sparkles
            for i, (sx, sy) in enumerate(MEADOW_SPARKS):
                a = int(150 + 105 * (0.5 + 0.5 * math.sin(g['t'] * 2.4 + i * 1.7)))
                col = (240, 196, 70, a)
                pixel_rect(ox + sx - 2, sy - 9, 4, 18, col)
                pixel_rect(ox + sx - 9, sy - 2, 18, 4, col)
        for ox in _tiles(scroll, 0.15, 800):           # MEADOW: far round trees
            for tx, r in ((90, 62), (420, 80), (700, 56)):
                pixel_round_tree(ox + tx, GROUND - 4, r, th, far=True)

    # Drifting pixel clouds (smoke on the volcano map).
    for ox in _tiles(scroll, 0.12, 900, drift=g['t'] * 5):
        for cx, cy in ((150, 440), (520, 505), (736, 430)):
            pixel_cloud(ox + cx, cy, th['cloud'], th['cloud_hi'])

    if kind in ('sunset', 'volcano'):
        for ox in _tiles(scroll, 0.30, 700):
            for mx, w, h in ((200, 300, 110), (520, 340, 140)):
                pixel_mountain(ox + mx, GROUND - 4, w, h, th['mt_near'])
    if kind == 'night':
        for ox in _tiles(scroll, 0.30, 600):
            for bx in (40, 330, 510):
                pixel_rect(ox + bx, GROUND - 2, 60, 25, th['bush'])
                pixel_rect(ox + bx + 13, GROUND + 23, 38, 22, th['bush_hi'])
    if kind == 'forest':                               # FOREST: near pines + rocks
        for ox in _tiles(scroll, 0.30, 700):
            for tx, h in ((40, 340), (235, 230), (560, 300)):
                pixel_pine(ox + tx, GROUND - 4, h, th)
            for rx, w, h in ((380, 52, 66), (650, 40, 48)):
                pixel_rock(ox + rx, GROUND - 4, w, h, th)
    elif kind == 'meadow':                             # MEADOW: pipes, mushrooms, keys
        for ox in _tiles(scroll, 0.30, 800):
            pixel_pipe(ox + 150, GROUND - 4, 150, th)
            pixel_pipe(ox + 560, GROUND - 4, 105, th)
            pixel_mushroom(ox + 340, GROUND - 4, 1.4)
            pixel_mushroom(ox + 700, GROUND - 4, 1.0)
            pixel_key(ox + 330, 440 + math.sin(g['t'] * 2 + ox) * 6)
    if kind == 'volcano':  # rising embers
        for ox in _tiles(scroll, 0.10, 900):
            for i in range(28):
                x = ox + (i * 97) % 900 + math.sin(g['t'] * 1.3 + i) * 14
                y = (i * 53 + g['t'] * (35 + (i % 4) * 14)) % 640 - 20
                a = int(120 + 100 * math.sin(g['t'] * 3 + i))
                pixel_rect(x, y, 4, 4, (255, 150, 60, max(0, min(255, a))))


def draw_ground():
    th = current_theme()
    cam = g['dcam']
    left_w, right_w = cam + view['world_left'], cam + view['world_right']
    bottom = min(0, view['world_bottom'])
    # Pits (lava on the volcano map).
    for a, b in HOLES:
        l, r = max(a, left_w), min(b, right_w)
        if r > l:
            arcade.draw_lrbt_rectangle_filled(l-cam, r-cam, bottom, GROUND,
                                             th['pit'])
    for a, b in SEGMENTS:
        # The opening grass segment extends into the additional side scenery.
        start = min(a, left_w) if a == 0 and cam <= 0 else a
        l, r = max(start, left_w), min(b, right_w)
        if r <= l:
            continue
        sl, sr = l - cam, r - cam
        arcade.draw_lrbt_rectangle_filled(sl, sr, bottom, GROUND-12, th['soil'])
        arcade.draw_lrbt_rectangle_filled(sl, sr, bottom, 28, th['soil_dark'])
        arcade.draw_lrbt_rectangle_filled(sl, sr, GROUND-12, GROUND, th['grass'])
        arcade.draw_lrbt_rectangle_filled(sl, sr, GROUND-4, GROUND, th['grass_light'])
    # Pixel "bricks" in the soil (deterministic, so they don't flicker).
    first, last = int(left_w // 60) - 1, int(right_w // 60) + 2
    for i in range(first, last):
        px = i*60 + (i*37) % 40
        if ground_exists(px-8) and ground_exists(px+8):
            py = 34 + (i*53) % 44
            pixel_rect(px-cam, py, 18, 9, th['soil_dot'])
            pixel_rect(px-cam+22, py-20 if py > 40 else py+12, 12, 8,
                       th['soil_dark'])
    # Grass tufts: 3 thin blades each, swaying gently in the "wind".
    first, last = int(left_w // 34) - 1, int(right_w // 34) + 2
    for i in range(first, last):
        tx = i*34 + (i*17) % 20
        if ground_exists(tx-10) and ground_exists(tx+10):
            sx = tx - cam
            sway = math.sin(g['t']*2.2 + tx*0.05) * 2.5
            scale = 0.8 + (i % 3) * 0.2
            for dx, bh, lean, col in ((-5, 9, -3, th['tuft']),
                                      (0, 14, 0, th['grass_light']),
                                      (5, 10, 3, th['tuft'])):
                bx = sx + dx
                top = bh * scale
                tip = bx + lean + sway * (top / 14)
                arcade.draw_triangle_filled(bx-2, GROUND-1, tip, GROUND-1+top,
                                            bx+2, GROUND-1, col)


def draw_platforms():
    """PLATFORMS: floating islands painted with the current map's soil/grass."""
    th = current_theme()
    cam = g['dcam']
    for p in g['platforms']:
        w = p['w']
        l = p['x'] - w / 2 - cam
        top = p['y']
        if l + w < view['world_left'] - 30 or l > view['world_right'] + 30:
            continue
        pixel_rect(l + 4, top - PLAT_THICK - 4, w - 8, 4, th['soil_dark'])   # tapered base
        pixel_rect(l, top - PLAT_THICK, w, PLAT_THICK - 8, th['soil'])
        pixel_rect(l, top - PLAT_THICK, w, 5, th['soil_dark'])
        for k in range(3):                                            # soil bricks
            pixel_rect(l + 14 + k * (w - 40) / 2, top - 21, 12, 6, th['soil_dot'])
        pixel_rect(l + 10, top - PLAT_THICK - 12, 10, 8, th['soil_dark'])    # hanging bits
        pixel_rect(l + w - 28, top - PLAT_THICK - 9, 8, 6, th['soil_dark'])
        pixel_rect(l - 3, top - 9, w + 6, 9, th['grass'])
        pixel_rect(l - 3, top - 3, w + 6, 3, th['grass_light'])
        for k in range(int(w // 22)):                                 # grass tufts
            pixel_rect(l + 6 + k * 22, top, 4, 5, th['tuft'])


# ---------------------------------------------------------------------------
# BOSS: KING WATERMELON floats on the right of the screen, throws junk at you
# (a red mark shows where it will land - dodge it!) and loses 1 HP for every
# fruit you eat. HP = goal - score, so eating the goal defeats him.
# ---------------------------------------------------------------------------
BOSS_Y, BOSS_G = 400, 700


def is_boss_level():
    return LEVELS[max(0, min(g['level'], NUM_LEVELS - 1))].get('boss', False)


def boss_screen_pos():
    slide = (g['boss_intro'] / 2.6) ** 2 * 320 if g['boss_intro'] > 0 else 0.0
    return (view['world_right'] - 110 + slide,
            BOSS_Y + math.sin(g['t'] * 2.0) * 8)


def boss_remaining():
    return max(0, g['goal'] - g['score'])


def boss_hit(power=1):
    g['boss_hurt'] = 0.45
    bx, by = boss_screen_pos()
    spawn_burst(g['cam'] + bx, by, (255, 120, 90), 10 + 5 * power, speed=300)


def spawn_boss_junk(offset):
    cfg = LEVELS[g['level']]
    bx, by = boss_screen_pos()
    x0, y0 = g['cam'] + bx - 25, by - 25
    tx = max(60.0, min(WORLD_W - 60.0, g['x'] + g['vx'] * 0.45 + offset))
    fl = floor_under(g['x'], g['y'])
    fl = GROUND if fl is None else fl
    ty = fl + 26
    ratio = boss_remaining() / max(1, g['goal'])
    flight = 1.25 if ratio > 0.66 else (1.1 if ratio > 0.33 else 1.0)
    g['boss_hazards'].append({
        'x': x0, 'y': y0, 'vx': (tx - x0) / flight,
        'vy': (ty - y0 + 0.5 * BOSS_G * flight * flight) / flight,
        'tx': tx, 'fl': fl, 'ty': ty, 't': 0.0, 'T': flight, 'rest': 0.0,
        'ang': random.uniform(0, 360), 'spin': random.choice((-1, 1)) * random.uniform(200, 360),
        'kind': random.choice(BAD_ITEMS[:cfg['bad_n']])})
    g['boss_roar'] = 0.35


def update_boss(dt):
    g['boss_hurt'] = max(0.0, g['boss_hurt'] - dt)
    g['boss_roar'] = max(0.0, g['boss_roar'] - dt)
    if g['boss_intro'] <= 0:
        g['boss_throw'] -= dt
        if g['boss_throw'] <= 0:
            ratio = boss_remaining() / max(1, g['goal'])
            gap, n = (3.4, 1) if ratio > 0.66 else ((2.6, 1) if ratio > 0.33 else (2.1, 2))
            g['boss_throw'] = gap
            spawn_boss_junk(random.uniform(-90, 90))
            if n == 2:                                   # last third: two at once
                spawn_boss_junk(random.choice((-170, 170)))
    keep = []
    for h in g['boss_hazards']:
        if h['t'] < h['T']:
            h['t'] += dt
            h['vy'] -= BOSS_G * dt
            h['x'] += h['vx'] * dt
            h['y'] += h['vy'] * dt
            if h['t'] >= h['T']:
                h['x'], h['y'] = h['tx'], h['ty']
        else:
            h['rest'] += dt
        h['ang'] += h['spin'] * dt
        if h['rest'] > 0.4:
            continue
        if (g['invulnerable'] <= 0 and abs(h['x'] - g['x']) < 44
                and g['y'] - 8 < h['y'] < g['y'] + 125):
            spawn_burst(h['x'], h['y'], (150, 150, 165), 10, speed=220)
            lose_heart('BOSS JUNK! -1 HEART')
            continue
        keep.append(h)
    g['boss_hazards'] = keep


def disc_rows(cx, cy, rx, ry, step=4):
    for j in range(-int(ry // step), int(ry // step)):
        yy = j * step
        f = 1 - ((yy + step / 2) / ry) ** 2
        if f <= 0:
            continue
        half = round(rx * math.sqrt(f) / step) * step
        if half > 0:
            yield cy + yy, half


def draw_boss():
    if not is_boss_level() or g['screen'] not in ('playing', 'win', 'lose'):
        return
    dead = g['screen'] == 'win'
    hurt = g['boss_hurt'] > 0
    flash = hurt and int(g['t'] * 28) % 2 == 0
    bx, by = boss_screen_pos()
    if hurt:
        bx += math.sin(g['t'] * 60) * 4

    def col(c):
        if flash:
            return mix(c, WHITE, 0.65)
        return mix(c, (150, 150, 150), 0.5) if dead else c

    rx, ry = 56, 50
    for y, half in disc_rows(bx, by, rx + 4, ry + 4):          # dark outline
        pixel_rect(bx - half, y, half * 2, 4, col((20, 36, 40)))
    for y, half in disc_rows(bx, by, rx, ry):                   # striped rind
        pixel_rect(bx - half, y, half * 2, 4, col((34, 110, 52)))
        for sc in (-40, -20, 0, 20, 40):
            if abs(sc) + 4 <= half:
                pixel_rect(bx + sc - 4, y, 8, 4, col((84, 176, 70)))
    ey = by + 8
    for sx in (-1, 1):
        ex = bx + sx * 24
        if dead:                                                 # X eyes
            for k in range(-2, 3):
                pixel_rect(ex + k * 5 - 2.5, ey + k * 5 - 2.5, 5, 5, col((30, 20, 40)))
                pixel_rect(ex + k * 5 - 2.5, ey - k * 5 - 2.5, 5, 5, col((30, 20, 40)))
        else:
            pixel_rect(ex - 12, ey - 10, 24, 20, col(WHITE))
            pixel_rect(ex - 12 + (2 if sx < 0 else 0) - 1, ey - 6, 9, 12, col((190, 30, 50)))
            hi = 18 if sx < 0 else 10                            # angry eyebrows
            lo = 10 if sx < 0 else 18
            pixel_rect(ex - 14, ey + hi, 12, 6, col((20, 36, 40)))
            pixel_rect(ex - 2, ey + lo, 12, 6, col((20, 36, 40)))
    mh = 28 if (g['boss_roar'] > 0 and not dead) else 20
    pixel_rect(bx - 31, by - 37, 62, mh + 2, col((60, 10, 30)))
    pixel_rect(bx - 27, by - 35, 54, mh - 2, col((230, 60, 80)))
    for i in range(5):
        pixel_rect(bx - 25 + i * 11.5, by - 37 + mh - 8, 7, 8, col(WHITE))
    pixel_crown(bx, by + ry + 2, 60, 32)
    if not dead:                                                 # HP bar
        bw, bh = 132, 12
        bl, bb = bx - bw / 2, by + ry + 38
        pixel_rect(bl - 3, bb - 3, bw + 6, bh + 6, (14, 8, 34))
        seg = (bw - 2 * (g['goal'] - 1)) / g['goal']
        for i in range(g['goal']):
            on = i < boss_remaining()
            pixel_rect(bl + i * (seg + 2), bb, seg, bh,
                       (230, 60, 70) if on else (62, 46, 108))
            if on:
                pixel_rect(bl + i * (seg + 2), bb + bh - 4, seg, 3, (255, 140, 140))
        txt('KING WATERMELON', bx + 1, bb + bh + 9, (14, 8, 34), 9, 'center', 'center', True)
        txt('KING WATERMELON', bx, bb + bh + 10, (255, 222, 133), 9, 'center', 'center', True)


def draw_boss_junk():
    if g['screen'] != 'playing':
        return
    cam = g['dcam']
    for h in g['boss_hazards']:
        sx = h['x'] - cam
        if h['t'] < h['T']:                                      # landing warning
            k = h['t'] / h['T']
            pulse = 0.5 + 0.5 * math.sin(g['t'] * 18)
            arcade.draw_ellipse_filled(h['tx'] - cam, h['fl'] + 4, 70 - 26 * k, 14,
                                       (235, 60, 70, int(70 + 120 * k * (0.6 + 0.4 * pulse))))
            txt('!', h['tx'] - cam, h['fl'] + 26, (255, 222, 133), 16, 'center', 'center', True)
        alpha = 255 if h['t'] < h['T'] else int(255 * (1 - h['rest'] / 0.4))
        blit(tex[(h['kind'], 'item', False)], sx, h['y'], ITEM_SIZE * 0.95, ITEM_SIZE * 0.95,
             alpha=max(0, alpha), angle=h['ang'])


def draw_monster():
    cam = g['dcam']
    sx = g['x'] - cam
    mot = char_motion()                      # CHAR_MOTION: per-character feel
    airborne = standing_on() is None                 # PLATFORMS: any surface counts
    moving = abs(g['vx']) > 20 and not airborne
    hop = 0.0
    if g['screen'] == 'win':
        name = 'win'
        hop = abs(math.sin(g['t'] * WIN_HOP_SPEED)) * 14
    elif g['screen'] == 'lose':
        name = 'game_over'
    elif g['mouth'] == 'OPEN':
        name = 'mouth_open'
    elif moving:
        name = ('walk_1', 'walk_2', 'walk_3', 'walk_4')[int(g['walk_phase']) % 4]
    else:
        name = 'idle'

    playing = g['screen'] == 'playing'

    # Procedural step motion: works with ANY pose, so a character that is
    # walking with its mouth OPEN still hops / waddles instead of sliding.
    step_angle = 0.0
    step_squash = 0.0
    if moving and playing:
        ph = g['walk_phase'] * math.pi / 2          # one full stride = 4 frames
        step = abs(math.sin(ph))                    # 0..1 on every footfall
        hop += step * mot['step_hop']
        step_angle = math.sin(ph) * mot['waddle']   # rock left / right
        step_squash = (step - 0.5) * mot['squash']

    # Hovering characters (pink bat) float above the ground and bob.
    float_y = 0.0
    if mot['hover'] and g['screen'] not in ('win', 'lose'):
        fast = moving and playing
        wobble = math.sin(g['t'] * (9.0 if fast else 2.6))
        float_y = mot['hover'] + wobble * mot['float_amp'] * (1.0 if fast else 0.7)
    elif mot['hover']:
        float_y = mot['hover'] * 0.6 + math.sin(g['t'] * 2.6) * mot['float_amp']

    # Squash & stretch: stretch while rising/falling, squash on landing.
    stretch = 0.0
    if airborne:
        stretch = max(-0.10, min(0.14, g['vy'] / JUMP_SPEED * 0.14))
    land = g['land']
    if moving:
        bob = abs(math.sin(g['walk_phase'] * math.pi)) * 0.035
    else:
        bob = math.sin(g['t'] * 2.2) * 0.012
    scale_y = 1 + stretch - 0.16*land + bob + step_squash
    scale_x = 1 - stretch*0.6 + 0.20*land - bob*0.4 - step_squash*0.6
    w, h = MONSTER_GAME[0]*scale_x, MONSTER_GAME[1]*scale_y
    # Every monster PNG has different empty space around it, so measure the
    # visible feet (idle pose = stable baseline) and stand them ON the ground.
    feet_k = sprite_bounds(current_character(), 'idle')[3]   # visible bottom (0..1)
    feet = g['y'] + 1 + hop + float_y
    cy = feet - (1 - feet_k) * h + h/2

    # Soft shadow on the ground (shrinks as the monster rises / hovers).
    shadow_floor = floor_under(g['x'], g['y'])      # PLATFORMS: shadow on the surface below
    if shadow_floor is not None:
        height_above = max(0.0, g['y'] - shadow_floor) + float_y * 0.8
        k = max(0.25, 1 - height_above / 380)
        arcade.draw_ellipse_filled(sx, shadow_floor-3, 92*k, 15*k, (40, 24, 75, 70))

    flip = (g['facing'] < 0) == ART_FACES_RIGHT
    alpha = 255
    if g['screen'] == 'playing' and g['invulnerable'] > 0 \
            and int(g['animation_time'] * 18) % 2:
        alpha = 110
    lean = max(-6.0, min(6.0, -g['vx'] / SPEED * 4.0))
    blit(character_tex(name, 'game', flip), sx, cy, w, h, alpha=alpha,
         angle=lean + step_angle)


# ---------------------------------------------------------------------------
# HUD: top bar = [hearts panel]   [LEVEL panel]   [fruit progress panel]
# ---------------------------------------------------------------------------
HUD_H = 66
HUD_BACK = (110, 40)          # BACK button size inside the top bar


def hud_back_pos():
    """BACK sits at the top-left of the bar (follows the screen edge)."""
    return view['world_left'] + 14 + HUD_BACK[0] / 2, H - HUD_H / 2 + 2


def hud_panel(left, bottom, width, height):
    """Dark pixel panel: drop shadow, light border, inner fill, top gloss."""
    pixel_rect(left + 3, bottom - 4, width, height, (14, 8, 34))
    pixel_rect(left, bottom, width, height, (112, 78, 170))
    pixel_rect(left + 3, bottom + 3, width - 6, height - 6, (30, 18, 62))
    pixel_rect(left + 3, bottom + height - 7, width - 6, 3, (58, 40, 106))


def hud_text(msg, x, y, color, size, anchor_x='left'):
    """Text with a dark drop shadow so it stays readable on any panel."""
    txt(msg, x + 2, y - 2, (14, 8, 34), size, anchor_x, 'center', True)
    txt(msg, x, y, color, size, anchor_x, 'center', True)


def hud_bar_body():
    """Solid top bar (shared by the game HUD and the SAY THE FRUIT screen)."""
    L, R = view['world_left'], view['world_right']
    top = max(H, view['world_top'])
    bar_b = H - HUD_H
    arcade.draw_lrbt_rectangle_filled(L, R, bar_b - 9, bar_b, (0, 0, 0, 55))
    pixel_rect(L, bar_b, R - L, top - bar_b, (40, 24, 75))
    pixel_rect(L, bar_b, R - L, 6, (20, 10, 45))            # thick dark bottom edge
    pixel_rect(L, bar_b + 6, R - L, 2, (70, 46, 122))       # soft step above it
    pixel_rect(L, H - 3, R - L, 3, (132, 96, 196))          # light top line


def draw_hud():
    L, R = view['world_left'], view['world_right']
    cy = H - HUD_H / 2 + 2

    # --- bar body: solid, light line on top, thick dark edge + shadow below ---
    hud_bar_body()

    # --- hearts panel (left) ---
    hl = L + 14 + HUD_BACK[0] + 14          # right next to the BACK button
    hud_panel(hl, cy - 23, 152, 46)
    for i in range(3):
        hx = hl + 32 + i * 44
        full = i < g['hearts']
        beat = 1 + math.sin(g['t'] * 5 + i) * 0.04 if full else 1.0
        key = 'heart_full' if full else 'heart_empty'
        blit(tex[(key, 'hud', False)], hx, cy, 34 * beat, 34 * beat)
    lost_t = g.get('heart_lost_t', 0.0)
    if lost_t > 0:                      # the lost heart pops and fades away
        k = 1 - lost_t / 0.5
        size = 34 * (1 + 1.3 * k)
        hx = hl + 32 + g.get('heart_lost_i', 0) * 44
        blit(tex[('heart_full', 'hud', False)], hx, cy, size, size,
             alpha=int(255 * (1 - k)))

    # --- LEVEL panel (exact centre of the screen) ---
    mid = (L + R) / 2
    boss_lvl = LEVELS[max(0, min(g['level'], NUM_LEVELS - 1))].get('boss', False)
    if boss_lvl:                               # BOSS: pulsing red frame around the panel
        pulse = 0.5 + 0.5 * math.sin(g['t'] * 5)
        pixel_rect(mid - 57, cy - 28, 114, 56, (235, 60, 70, int(110 + 120 * pulse)))
    hud_panel(mid - 52, cy - 23, 104, 46)      # same height as the other panels
    if boss_lvl:
        hud_text(f'LEVEL {g["level"] + 1}', mid, cy + 11, (255, 222, 133), 10, 'center')
        hud_text('BOSS', mid, cy - 6, (255, 96, 104), 20, 'center')
    else:
        hud_text('LEVEL', mid, cy + 10, (255, 222, 133), 10, 'center')
        hud_text(str(g['level'] + 1), mid, cy - 7, WHITE, 19, 'center')

    # --- fruit progress panel (right) ---
    pw = 284
    fl = R - 14 - pw
    hud_panel(fl, cy - 23, pw, 46)
    lvl = max(0, min(g['level'], NUM_LEVELS - 1))
    pop = max(0.0, g.get('fruit_pop', 0.0)) / 0.3
    icon = 34 * (1 + 0.4 * math.sin(pop * math.pi))
    blit(tex[(LEVELS[lvl]['fruit'], 'item', False)], fl + 27, cy, icon, icon)
    goal = max(1, g['goal'])
    bar_l, bar_w, bar_h = fl + 52, 140, 16
    pixel_rect(bar_l - 3, cy - bar_h / 2 - 3, bar_w + 6, bar_h + 6, (14, 8, 34))
    gap = 3
    seg_w = (bar_w - gap * (goal - 1)) / goal
    for i in range(goal):
        sx0 = bar_l + i * (seg_w + gap)
        if i < g['score']:
            pixel_rect(sx0, cy - bar_h / 2, seg_w, bar_h, (96, 184, 78))
            pixel_rect(sx0, cy + 1, seg_w, 5, (170, 232, 130))
        else:
            pixel_rect(sx0, cy - bar_h / 2, seg_w, bar_h, (62, 46, 108))
    done = g['score'] >= goal
    hud_text(f'{min(g["score"], goal)}/{goal}', bar_l + bar_w + 14, cy,
             (255, 222, 133) if done else WHITE, 18)

    # --- status text: hidden while everything works (D toggles it) ---
    status_map = {
        'loading': 'Loading camera...',
        'nomodel': 'Add best.pt beside this file',
        'unavailable': 'Camera unavailable - press C to retry',
        'off': 'Camera off - press C to start',
    }
    cam_ready = g['cam_status'] == 'ready'
    hand_ready = g['hand_status'] == 'READY'
    status = 'MOUTH: ' + g['mouth'] if cam_ready else status_map.get(g['cam_status'], '')
    fresh = time.monotonic() - g['hand_updated'] < HAND_STALE_SECONDS
    direction = g['hand_gesture'] if fresh else 'STOP'
    hand_msg = 'HAND: ' + direction if hand_ready else 'HAND: ' + g['hand_status']
    if g['show_debug'] or not cam_ready or not hand_ready:
        line = status + '   ' + hand_msg
        pill_w = 10 + len(line) * 8.2
        arcade.draw_lrbt_rectangle_filled(14, 14 + pill_w, 10, 34,
                                         (30, 18, 62, 190))
        txt(line, 22, 22, WHITE, 12, anchor_y='center', bold=True)

    if g['boss_intro'] > 0 and g['screen'] == 'playing':      # BOSS: warning intro
        flash = 0.5 + 0.5 * math.sin(g['t'] * 14)
        arcade.draw_lrbt_rectangle_filled(L, R, view['world_bottom'], view['world_top'],
                                          (210, 30, 50, int(22 + 40 * flash)))
        px, py = W / 2, H / 2 + 20
        pixel_rect(px - 196, py - 6, 392, 106, (14, 8, 34))
        hud_panel(px - 190, py, 380, 98)
        shake = math.sin(g['t'] * 40) * 2 if g['boss_intro'] > 2.0 else 0
        hud_text('BOSS LEVEL!', px + shake, py + 62, (255, 96, 104), 38, 'center')
        hud_text('Dodge the junk the boss throws!', px, py + 34,
                 (255, 222, 133), 13, 'center')
        hud_text('Every fruit you eat hurts him!', px, py + 15,
                 (255, 222, 133), 13, 'center')

    theme_text = current_theme()['text']   # MAPS: readable on every map
    if g['screen'] == 'playing':
        bx, by = hud_back_pos()
        draw_back_button(bx, by, *HUD_BACK)
    if g['banner_time'] > 0:
        drop = max(0.0, g['banner_time'] - 1.2) * 220
        txt(g['banner'], W/2, H-HUD_H-34+drop, theme_text, 21,
            anchor_x='center', bold=True)


def view_cam():
    s = g['shake']
    off = random.uniform(-1, 1) * s * 28 if s > 0 else 0
    return round(g['cam'] + off)


def draw_world():
    g['dcam'] = view_cam()
    draw_map_background(g['dcam'])      # MAPS: themed background per level
    draw_boss()                         # BOSS
    draw_ground()
    draw_platforms()                    # PLATFORMS
    cam = g['dcam']
    for thing in g['objects']:
        if thing['taken'] and thing['pop'] <= 0:
            continue
        x = thing['x'] - cam
        if not (view['world_left']-80 < x < view['world_right']+80):
            continue
        wob = math.sin(g['t']*3 + thing['x'])
        king = thing.get('king', False)               # BOSS: golden king fruit
        base = ITEM_SIZE * (1.5 if king else 1.0)
        texture = tex[(thing['kind'], 'big' if king else 'item', False)]
        fl = thing.get('floor', GROUND)               # PLATFORMS: shadow on its own floor
        if thing['taken']:
            k = 1 - thing['pop'] / 0.3
            s = 1 + 0.7*k
            blit(texture, x, thing['y'] + k*24, base*s, base*s,
                 alpha=int(255*(1-k)))
        else:
            arcade.draw_ellipse_filled(x, fl+3, (38 - wob*4) * (1.5 if king else 1.0), 9,
                                       (40, 24, 75, 55))
            sc = 1 + wob*0.03
            cy_item = thing['y'] + wob*6
            if king:
                pulse = 0.5 + 0.5 * math.sin(g['t'] * 4)
                arcade.draw_circle_filled(x, cy_item, 54 + pulse * 6,
                                          (255, 214, 90, int(40 + 36 * pulse)))
                arcade.draw_circle_filled(x, cy_item, 40, (255, 232, 140, 40))
            blit(texture, x, cy_item, base*sc, base*sc)
            if king:
                pixel_crown(x, cy_item + base * 0.34, 34, 19)
                draw_pixel_sparkle(x + 40, cy_item + 26, thing['x'], 0.9)
                draw_pixel_sparkle(x - 38, cy_item - 8, thing['x'] + 2, 0.7)
    draw_boss_junk()                    # BOSS: flying junk + landing marks
    draw_monster()
    draw_particles()
    draw_hud()


def draw_title(cx, cy):
    cy = cy + math.sin(g['t']*1.5) * 3
    txt('FRUITY MONSTER', cx+2, cy-3, (255, 255, 255), 38,
        anchor_x='center', anchor_y='center', bold=True)
    txt('FRUITY MONSTER', cx, cy, INK, 38, anchor_x='center',
        anchor_y='center', bold=True)


def draw_menu_base(monster_pose='idle', show_title=True):
    draw_background(0)
    # Decorative single flat grass strip at the bottom; no blank dark region.
    arcade.draw_lrbt_rectangle_filled(view['world_left'], view['world_right'], 0, 60, (179, 135, 103))
    arcade.draw_lrbt_rectangle_filled(view['world_left'], view['world_right'], 60, 75, (128, 195, 88))
    arcade.draw_lrbt_rectangle_filled(view['world_left'], view['world_right'], 70, 75, GRASS_LIGHT)
    # Monster on the LEFT, breathing (or hopping when happy).
    s = math.sin(g['t'] * 2.0)
    if monster_pose in ('happy', 'win'):
        bob = abs(math.sin(g['t'] * WIN_HOP_SPEED)) * 16
    else:
        bob = s * 4
    w = MONSTER_MENU[0] * (1 - 0.010*s)
    h = MONSTER_MENU[1] * (1 + 0.015*s)
    bottom = 150 - MONSTER_MENU[1]/2 + bob
    arcade.draw_ellipse_filled(182, 78, 190 - bob*2, 20, (40, 24, 75, 60))
    blit(character_tex(monster_pose, 'menu'), 182, bottom + h/2, w, h)
    if show_title:
        draw_title(570, 488)


# ---------------------------------------------------------------------------
# New introductory UI: START -> CHARACTER SELECT -> CHOOSE GAME
# Existing mechanics, scores, hand tracking, YOLO and speech are unchanged.
# All four sprite sets are loaded from their corresponding folders.
# ---------------------------------------------------------------------------
CHARACTER_CHOICES = (
    ('PURPLE', (160, 87, 220)),
    ('GREEN', (106, 195, 160)),
    ('ORANGE', (245, 148, 104)),
    ('PINK', (240, 116, 184)),
)


def pixel_button(cx, cy, width, height, label, enabled=True):
    """Pixel-art themed clickable area; hit-test uses cx/cy/width/height."""
    hover = enabled and hit(*g['mouse'], cx, cy, width, height)
    left, bottom = cx - width/2, cy - height/2
    shadow = (70, 34, 100)
    if enabled:
        edge = (239, 113, 147) if hover else (215, 91, 135)
        surface = (255, 210, 95) if hover else (247, 193, 91)
        shine = (255, 234, 139)
        text_color = (96, 39, 110)
    else:
        edge, surface = (110, 96, 140), (139, 128, 157)
        shine, text_color = (165, 154, 184), (80, 66, 108)
    pixel_rect(left + 7, bottom - 8, width - 14, height, shadow)
    pixel_rect(left, bottom + 8, width, height - 16, edge)
    pixel_rect(left + 8, bottom, width - 16, height, edge)
    pixel_rect(left + 10, bottom + 8, width - 20, height - 16, surface)
    pixel_rect(left + 18, bottom + height - 17, width - 36, 6, shine)
    txt(label, cx, cy, text_color, 20, 'center', 'center', True)


def draw_back_button(cx=90, cy=35, width=138, height=46):
    """Clickable pixel BACK button: yellow face, purple shadow, no pink trim."""
    hover = hit(*g['mouse'], cx, cy, width, height)
    left, bottom = cx - width / 2, cy - height / 2
    # Purple drop shadow; no pink border on BACK.
    pixel_rect(left + 5, bottom - 6, width - 10, height, (70, 34, 100))
    # Solid yellow button, with a small lighter highlight on top.
    surface = (255, 210, 95) if hover else (247, 193, 91)
    pixel_rect(left, bottom, width, height, surface)
    pixel_rect(left + 10, bottom + height - 10, width - 20, 5,
               (255, 234, 139))
    txt('BACK', cx, cy, (96, 39, 110), 20, 'center', 'center', True)


def draw_pixel_sparkle(cx, cy, phase=0.0, strength=1.0):
    """Animated pixel-star glint; Arcade shapes only, no extra PNG needed."""
    pulse = (0.5 + 0.5 * math.sin(g['t'] * 3.3 + phase)) ** 2
    brightness = pulse * strength
    if brightness < 0.025:
        return
    # Diffuse halo, then sharp pixel cross so the retro silhouette stays crisp.
    arcade.draw_circle_filled(cx, cy, 17, (255, 227, 105, int(43 * brightness)))
    arcade.draw_circle_filled(cx, cy, 9, (255, 247, 186, int(67 * brightness)))
    col = (255, 252, 210, int(240 * brightness))
    pixel_rect(cx - 2, cy - 11, 4, 22, col)
    pixel_rect(cx - 11, cy - 2, 22, 4, col)
    pixel_rect(cx - 2, cy - 2, 4, 4, WHITE)


def draw_pixel_scene(start_screen=False):
    """Original Arcade-rendered retro decoration inspired by the reference."""
    pixel_rect(view['world_left'], view['world_bottom'],
               view['world_right'] - view['world_left'],
               view['world_top'] - view['world_bottom'], (24, 31, 78))
    # Pixel stars: twinkling halos on the START screen, original pixels preserved.
    for i, (sx, sy) in enumerate(((80, 550), (225, 372), (835, 545),
                                  (790, 322), (128, 275), (625, 405))):
        if start_screen:
            sparkle = 0.5 + 0.5 * math.sin(g['t'] * 2.7 + i * 1.5)
            arcade.draw_circle_filled(sx, sy, 25,
                                      (255, 223, 106, int(50 * sparkle)))
            arcade.draw_circle_filled(sx, sy, 16,
                                      (255, 241, 175, int(72 * sparkle)))
        pixel_rect(sx - 3, sy - 13, 6, 26, (255, 216, 97))
        pixel_rect(sx - 13, sy - 3, 26, 6, (255, 216, 97))
        pixel_rect(sx - 4, sy + 10, 8, 5, (255, 241, 176))
        if start_screen:
            draw_pixel_sparkle(sx, sy, i * 1.5, strength=0.7)
    # Pixel clouds
    for cx, cy in ((150, 440), (736, 460)):
        for dx, dy, ww, hh in ((-50, 0, 102, 22), (-27, 18, 75, 24),
                                (-6, 36, 45, 18)):
            pixel_rect(cx + dx, cy + dy, ww, hh, (127, 89, 188))
        pixel_rect(cx - 38, cy + 8, 76, 11, (231, 140, 192))
    # Pixel ground
    scene_l = view['world_left']
    scene_w = view['world_right'] - scene_l
    pixel_rect(scene_l, 0, scene_w, 68, (147, 77, 74))
    pixel_rect(scene_l, 68, scene_w, 21, (97, 175, 78))
    pixel_rect(scene_l, 84, scene_w, 9, (150, 209, 88))
    for k in range(math.floor(scene_l / 32)*32, math.ceil(view['world_right'] / 32)*32, 32):
        pixel_rect(k + 6, 49, 19, 10, (175, 101, 75))
        pixel_rect(k + 2, 20, 14, 12, (119, 62, 80))
    # Small pixel bushes
    for bx in ((18, 675, 795) if start_screen else (18, 150, 675, 795)):
        pixel_rect(bx, 91, 60, 25, (69, 139, 94))
        pixel_rect(bx + 13, 115, 38, 22, (94, 177, 104))


def draw_start():
    draw_pixel_scene(start_screen=True)
    # Decorative 3 hearts, like the reference image.
    for i in range(3):
        hx = 396 + i * 56
        pixel_rect(hx - 14, 534, 11, 12, (235, 75, 100))
        pixel_rect(hx + 3, 534, 11, 12, (235, 75, 100))
        pixel_rect(hx - 18, 524, 36, 14, (235, 75, 100))
        pixel_rect(hx - 11, 512, 22, 13, (235, 75, 100))
        pixel_rect(hx - 4, 504, 8, 9, (235, 75, 100))
    # A soft animated title glow and traveling glints; lettering remains sharp.
    glow_strength = 0.5 + 0.5 * math.sin(g['t'] * 2.1)
    for words, yy, size, color, halo in (
        ('FRUITY', 414, 63, WHITE, (205, 132, 245)),
        ('MONSTER', 315, 67, (249, 223, 104), (255, 191, 96)),
    ):
        txt(words, W/2+5, yy-5, (62, 43, 103), size, 'center', 'center', True)
        txt(words, W/2, yy,
            (*halo, int(20 + 38 * glow_strength)), size + 2,
            'center', 'center', True)
        txt(words, W/2, yy, color, size, 'center', 'center', True)
    # Glints take turns passing across the title (no flat-picture animation).
    x_top = 290 + (g['t'] * 56) % 320
    x_bottom = 280 + ((g['t'] * 48 + 160) % 340)
    draw_pixel_sparkle(x_top, 441, phase=0.0, strength=0.85)
    draw_pixel_sparkle(x_bottom, 347, phase=1.2, strength=0.95)
    pixel_button(W/2, 194, 225, 70, 'START')
    txt('A FUN WAY TO PLAY & LEARN', W/2, 125, (236, 199, 244), 13,
        anchor_x='center', anchor_y='center', bold=True)
    # The start screen always shows the purple monster (not the chosen one).
    blit(character_tex('idle', character='purple'), 195, 155, 240, 255)


def character_ready(index):
    # All four characters have playable sets, loaded once at startup.
    return 0 <= index < len(CHARACTER_NAMES)


def draw_character_select():
    draw_pixel_scene()
    txt('CHOOSE YOUR CHARACTER', W/2, 525, WHITE, 29,
        anchor_x='center', anchor_y='center', bold=True)
    txt('Pick a character to begin', W/2, 479, (245, 218, 252), 15,
        anchor_x='center', anchor_y='center')
    for index, (label, shade) in enumerate(CHARACTER_CHOICES):
        character = CHARACTER_NAMES[index]
        cx = 150 + index * 200
        cy = 292
        hovered = hit(*g['mouse'], cx, cy, 172, 268)
        selected = current_character() == character
        frame = (255, 224, 112) if hovered or selected else shade
        pixel_rect(cx - 86 + 5, cy - 135 - 5, 172, 270, (13, 13, 42))
        pixel_rect(cx - 86, cy - 135, 172, 270, frame)
        pixel_rect(cx - 80, cy - 129, 160, 258, (61, 46, 116))
        # Character-specific idle previews; no placeholders or fake duplicates.
        blit(character_tex('idle', 'game', character=character),
             cx, cy + 30, 150, 164)
        txt('SELECT' if hovered else 'READY', cx, cy - 77,
            (255, 226, 115) if hovered else (168, 244, 161), 13,
            anchor_x='center', bold=True)
        txt(label, cx, cy - 111, WHITE, 13, 'center', 'center', True)
    draw_back_button(90, 33)


def draw_mode_select():
    draw_pixel_scene(start_screen=True)
    txt('CHOOSE YOUR GAME', W/2+4, 504-4, (62, 43, 103), 31,
        'center', 'center', True)
    txt('CHOOSE YOUR GAME', W/2, 504, WHITE, 31, 'center', 'center', True)
    txt('Pick a game to play', W/2, 462, (245, 218, 252), 15,
        'center', 'center')
    pixel_button(W/2, 379, 360, 70, "EAT OR DON'T EAT")
    pixel_button(W/2, 277, 360, 70, 'BREAK THE FRUIT')
    pixel_button(W/2, 175, 360, 70, 'SAY THE FRUIT')
    draw_back_button(90, 33)


# ---------------------------------------------------------------------------
# LEVEL SELECT (5 cards). Card centre x = 110 + 170*i, centre y = 285.
# ---------------------------------------------------------------------------
LV_CARD_W, LV_CARD_H, LV_CARD_Y = 150, 250, 285


def level_card_x(index):
    return 110 + 170 * index


def draw_star_shape(cx, cy, radius, color):
    """Filled 5-point star built from a polygon (no image needed)."""
    pts = []
    for k in range(10):
        r = radius if k % 2 == 0 else radius * 0.45
        a = math.pi / 2 + k * math.pi / 5
        pts.append((cx + math.cos(a) * r, cy + math.sin(a) * r))
    arcade.draw_polygon_filled(pts, color)


def draw_lock(cx, cy):
    """Pixel padlock."""
    pixel_rect(cx - 10, cy + 6, 4, 14, (190, 190, 205))      # shackle left
    pixel_rect(cx + 6, cy + 6, 4, 14, (190, 190, 205))       # shackle right
    pixel_rect(cx - 10, cy + 18, 20, 4, (190, 190, 205))     # shackle top
    pixel_rect(cx - 14, cy - 12, 28, 20, (247, 193, 91))     # body
    pixel_rect(cx - 2, cy - 8, 4, 10, (96, 39, 110))         # keyhole


def draw_level_card(index):
    cfg = LEVELS[index]
    unlocked = index < g['unlocked']
    boss = index == NUM_LEVELS - 1
    cx, cy = level_card_x(index), LV_CARD_Y
    hover = unlocked and hit(*g['mouse'], cx, cy, LV_CARD_W, LV_CARD_H)
    # Newest unlocked level gently bobs so the player knows where to go next.
    if unlocked and index == g['unlocked'] - 1 and g['stars'][index] == 0:
        cy += math.sin(g['t'] * 4) * 3
    if hover:
        cy += 6
    left, bottom = cx - LV_CARD_W / 2, cy - LV_CARD_H / 2
    frame = cfg['color'] if unlocked else (92, 70, 128)
    if hover:
        frame = tuple(min(255, c + 40) for c in frame)
    inner = (61, 46, 116) if unlocked else (38, 30, 78)

    pixel_rect(left + 5, bottom - 5, LV_CARD_W, LV_CARD_H, (13, 13, 42))   # shadow
    pixel_rect(left, bottom, LV_CARD_W, LV_CARD_H, frame)                  # frame
    pixel_rect(left + 6, bottom + 6, LV_CARD_W - 12, LV_CARD_H - 12, inner)

    # Title
    label = 'BOSS' if boss else f'LEVEL {index + 1}'
    txt(label, cx, cy + 105, WHITE if unlocked else (150, 140, 175), 15,
        'center', 'center', True)

    # Fruit (silhouette when locked)
    fruit_tex = tex[(cfg['fruit'], 'big', False)]
    size = BIG_ITEM_SIZE * (1.15 if boss else 1.0)
    if unlocked:
        bob = math.sin(g['t'] * 2.5 + index) * 4
        arcade.draw_ellipse_filled(cx, cy + 14, 52, 9, (13, 13, 42, 90))
        blit(fruit_tex, cx, cy + 58 + bob, size, size)
        if boss:                                        # BOSS: crown on the fruit
            pixel_crown(cx, cy + 58 + bob + size * 0.30, 38, 22)
    else:
        arcade.draw_texture_rect(fruit_tex, arcade.XYWH(cx, cy + 58, size, size),
                                 color=arcade.color.BLACK, alpha=170)
        draw_lock(cx, cy + 58)

    # Goal
    txt(f'EAT {cfg["goal"]}', cx, cy - 8, (255, 222, 133) if unlocked else (130, 120, 160),
        14, 'center', 'center', True)

    # Stars (earned = gold, otherwise grey)
    earned = g['stars'][index]
    for s in range(3):
        sx = cx - 34 + s * 34
        on = unlocked and s < earned
        draw_star_shape(sx, cy - 42, 13, (255, 216, 97) if on else (88, 78, 120))

    # Status
    if unlocked:
        txt('PLAY', cx, cy - 92, (168, 244, 161), 15, 'center', 'center', True)
    else:
        txt('LOCKED', cx, cy - 92, (150, 140, 175), 13, 'center', 'center', True)

    # Golden sparkle on the boss card
    if boss and unlocked:
        draw_pixel_sparkle(cx + 58, cy + 112, 0.5, 0.9)
        draw_pixel_sparkle(cx - 58, cy + 100, 2.1, 0.7)


def draw_level_select():
    draw_pixel_scene()
    txt('SELECT LEVEL', W/2, 525, WHITE, 29, anchor_x='center',
        anchor_y='center', bold=True)
    total = sum(g['stars'])
    txt(f'Stars: {total} / {NUM_LEVELS * 3}', W/2, 483, (245, 218, 252), 15,
        anchor_x='center', anchor_y='center')
    for index in range(NUM_LEVELS):
        draw_level_card(index)
    draw_back_button(90, 33)


def draw_menu():
    draw_menu_base()
    txt('Choose your adventure', 570, 407, INK, 19, anchor_x='center', bold=True)
    fancy_button(583, 317, 356, 69, "EAT OR DON'T EAT")
    fancy_button(583, 226, 356, 69, 'SAY THE FRUIT')
    txt('Move, collect and learn!', 583, 141, INK, 15, anchor_x='center')


# ---------------------------------------------------------------------------
# VOICE2: SAY THE FRUIT screen (night background + fruit progress bar on top)
# ---------------------------------------------------------------------------
def voice_listen_ready():
    return not (g['voice_busy'] or g['voice_wait'] > 0 or g['voice_finished'])


def draw_voice_bar():
    """Top bar with ALL fruits. A fruit lights up (and its segment fills)
    every time the player says it correctly."""
    L, R = view['world_left'], view['world_right']
    cy = H - HUD_H / 2 + 2
    hud_bar_body()

    n = len(FRUIT_TYPES)
    slot_w = 40                       # slimmer slots
    iw = slot_w - 6
    pad = 12                          # space between panel edge and first fruit
    pw = pad + n * slot_w + 80
    pl = R - 14 - pw
    hud_panel(pl, cy - 22, pw, 44)    # shorter panel: stays inside the top bar (inner = cy-19 .. cy+19)

    fill_k = 1.0 - max(0.0, g['voice_pop']) / 0.5     # newest fruit fills gradually
    pop_k = max(0.0, g['voice_pop']) / 0.5
    for i, name in enumerate(FRUIT_TYPES):
        sx = pl + pad + i * slot_w
        got = name in g['voice_got']
        current = (name == g['voice_fruit'] and not g['voice_finished'])
        if current:                                   # highlight the fruit being asked
            pixel_rect(sx, cy - 8, iw, 22, (74, 54, 134))
        size = 18.0                                   # small: stays inside the highlight
        if got and name == g['voice_last'] and pop_k > 0:
            size *= 1 + 0.15 * math.sin(pop_k * math.pi)
        blit(tex[(name, 'item', False)], sx + iw / 2, cy + 3,
             size, size, alpha=255 if got else 90)
        pixel_rect(sx, cy - 15, iw, 4, (62, 46, 108))
        if got:
            k = fill_k if (name == g['voice_last'] and pop_k > 0) else 1.0
            pixel_rect(sx, cy - 15, iw * k, 4, (96, 184, 78))
            pixel_rect(sx, cy - 13, iw * k, 2, (170, 232, 130))

    done = len(g['voice_got']) >= n
    hud_text(f'{len(g["voice_got"])}/{n}', pl + pad + n * slot_w + 38, cy,
             (255, 222, 133) if done else WHITE, 17, 'center')

    bx, by = hud_back_pos()
    draw_back_button(bx, by, *HUD_BACK)


VOICE_STARS = ((60, 480), (410, 300), (400, 465), (868, 330), (855, 470),
               (300, 500))


def voice_mic_x():
    """Mic is centred; it slides left to make room when SKIP is shown."""
    return MIC_X_SKIP if (g['voice_wrong'] and not g['voice_finished']) else MIC_X_SOLO


_bounds_cache = {}
VOICE_POSES = ('idle', 'happy', 'sad', 'win')
VOICE_FOOT_Y = 90          # where the monster's feet stand (grass top is 93)


def sprite_bounds(character, pose):
    """Visible part of a sprite as fractions (left, top, right, bottom) of its
    PNG. Every monster has different padding, so we measure instead of guess."""
    key = (character, pose)
    if key not in _bounds_cache:
        img = raw_image(pose, character=character)
        mask = img.getchannel('A').point(lambda a: 255 if a > 24 else 0)
        box = mask.getbbox() or (0, 0, img.width, img.height)
        _bounds_cache[key] = (box[0] / img.width, box[1] / img.height,
                              box[2] / img.width, box[3] / img.height)
    return _bounds_cache[key]


def draw_voice_scene():
    """Calm NIGHT scene: stars/cloud only in empty areas, no halos or circles."""
    L, R = view['world_left'], view['world_right']
    B, T = view['world_bottom'], view['world_top']
    pixel_rect(L, B, R - L, T - B, (24, 31, 78))
    for i, (sx, sy) in enumerate(VOICE_STARS):
        a = int(150 + 105 * (0.5 + 0.5 * math.sin(g['t'] * 2.7 + i * 1.5)))
        col = (255, 216, 97, a)
        pixel_rect(sx - 3, sy - 13, 6, 26, col)
        pixel_rect(sx - 13, sy - 3, 26, 6, col)
    pixel_cloud(150, 440, (127, 89, 188), (231, 140, 192))
    # ground
    pixel_rect(L, 0, R - L, 68, (147, 77, 74))
    pixel_rect(L, 68, R - L, 21, (97, 175, 78))
    pixel_rect(L, 84, R - L, 9, (150, 209, 88))
    for k in range(math.floor(L / 32) * 32, math.ceil(R / 32) * 32, 32):
        pixel_rect(k + 6, 49, 19, 10, (175, 101, 75))
        pixel_rect(k + 2, 20, 14, 12, (119, 62, 80))
    for bx in (18, 842):
        pixel_rect(bx, 91, 60, 25, (69, 139, 94))
        pixel_rect(bx + 13, 115, 38, 22, (94, 177, 104))


def draw_voice():
    draw_voice_scene()

    # ---- monster (left): feet on the grass, centred, whatever its size ----
    character = current_character()
    pose = 'win' if g['voice_finished'] else g['voice_reaction']
    s = math.sin(g['t'] * 2.0)
    bob = (abs(math.sin(g['t'] * WIN_HOP_SPEED)) * 16 if pose in ('happy', 'win')
           else s * 4)
    hover = (10 + math.sin(g['t'] * 2.6) * 3) if char_motion()['hover'] else 0.0
    mw = MONSTER_MENU[0] * (1 - 0.010 * s)
    mh = MONSTER_MENU[1] * (1 + 0.015 * s)
    l, t, r, bt = sprite_bounds(character, pose)
    cx = 200 - ((l + r) / 2 - 0.5) * mw                      # visible part centred
    cy = VOICE_FOOT_Y + hover + bob - (1 - bt) * mh + mh / 2  # feet on the ground
    blit(character_tex(pose, 'menu'), cx, cy, mw, mh)

    # ---- feedback bubble: sits just above THIS monster's head ----
    top_frac = min(sprite_bounds(character, p)[1] for p in VOICE_POSES)
    base_bottom = (VOICE_FOOT_Y + hover
                   - (1 - sprite_bounds(character, 'idle')[3]) * MONSTER_MENU[1])
    head_y = base_bottom + MONSTER_MENU[1] * (1 - top_frac)  # highest visible point
    bub_b = max(250.0, min(386.0, head_y + 22))              # 22 = room for the hop
    if g['voice_reaction'] == 'happy':
        feedback_color = (168, 244, 161)
    elif g['voice_reaction'] == 'sad':
        feedback_color = (255, 128, 148)
    else:
        feedback_color = WHITE
    message = g['voice_result'][:42]
    font_size = 18 if len(message) <= 24 else (14 if len(message) <= 34 else 11)
    hud_panel(40, bub_b, 320, 44)
    txt(message, 200, bub_b + 22, feedback_color, font_size,
        anchor_x='center', anchor_y='center', bold=True)

    # ---- the card on the right holds everything ----
    hud_panel(VP_L, VP_B, VP_W, VP_H)
    txt('SAY THE FRUIT', VP_X + 2, 483, (14, 8, 34), 24, 'center', 'center', True)
    txt('SAY THE FRUIT', VP_X, 485, WHITE, 24, 'center', 'center', True)

    # fruit on its own little plate
    pixel_rect(VP_X - 60, 360, 120, 100, (96, 68, 150))
    pixel_rect(VP_X - 57, 363, 114, 94, (52, 38, 108))
    if g['voice_finished']:
        txt('ALL FRUITS', VP_X, 428, (255, 222, 133), 17, 'center', 'center', True)
        txt('COMPLETE!', VP_X, 398, (255, 222, 133), 17, 'center', 'center', True)
        txt(f'You got {len(g["voice_got"])} / {len(FRUIT_TYPES)}', VP_X, 333,
            WHITE, 20, 'center', 'center', True)
    else:
        fy = 410 + math.sin(g['t'] * 2.5) * 4
        blit(tex[(g['voice_fruit'], 'big', False)], VP_X, fy, 76, 76)
        name = FRUIT_NAMES[g['voice_fruit']].upper()
        txt(name, VP_X + 2, 331, (14, 8, 34), 23, 'center', 'center', True)
        txt(name, VP_X, 333, (255, 222, 133), 23, 'center', 'center', True)

    # ---- STEP 1: listen ----
    ready = voice_listen_ready()
    pixel_button(VP_X, VOICE_LISTEN_Y, 320, 54, 'LISTEN TO THE WORD', enabled=ready)

    # ---- STEP 2: microphone (locked until the word was heard) ----
    mic_x, mic_y, mic_radius = voice_mic_x(), MIC_Y, MIC_R
    if g['voice_recording']:
        mic_color = (224, 83, 119)
        mic_caption = 'TAP TO STOP'
        pulse = (math.sin(g['t'] * 8) + 1) / 2
        arcade.draw_circle_filled(mic_x, mic_y, mic_radius + 5 + pulse * 9,
                                  (224, 83, 119, int(80 - pulse * 50)))
    elif g['voice_busy']:
        mic_color = (139, 128, 157)
        mic_caption = 'CHECKING...' if g['voice_task_kind'] == 'recognize' else 'LISTEN...'
    elif g['voice_finished']:
        mic_color = (139, 128, 157)
        mic_caption = 'ALL DONE'
    elif g['voice_wait'] > 0:
        mic_color = (139, 128, 157)
        mic_caption = 'WAIT...'
    elif not g['voice_heard']:
        mic_color = (110, 98, 140)
        mic_caption = 'LISTEN FIRST'
    else:
        mic_color = (155, 88, 210)
        mic_caption = 'TAP TO SPEAK'
        if hit(g['mouse'][0], g['mouse'][1], mic_x, mic_y, 78, 78):
            mic_color = (180, 116, 232)

    arcade.draw_circle_filled(mic_x, mic_y - 4, mic_radius + 2, (87, 42, 129))
    arcade.draw_circle_filled(mic_x, mic_y, mic_radius, mic_color)
    # Microphone pictogram from Arcade primitives (no extra image required).
    arcade.draw_lrbt_rectangle_filled(mic_x-7, mic_x+7, mic_y-2, mic_y+14, WHITE)
    arcade.draw_circle_filled(mic_x, mic_y+14, 7, WHITE)
    arcade.draw_circle_filled(mic_x, mic_y-2, 7, WHITE)
    arcade.draw_line(mic_x-14, mic_y+5, mic_x-14, mic_y-8, WHITE, 3)
    arcade.draw_line(mic_x+14, mic_y+5, mic_x+14, mic_y-8, WHITE, 3)
    arcade.draw_line(mic_x-14, mic_y-8, mic_x+14, mic_y-8, WHITE, 3)
    arcade.draw_line(mic_x, mic_y-8, mic_x, mic_y-17, WHITE, 3)
    arcade.draw_line(mic_x-9, mic_y-17, mic_x+9, mic_y-17, WHITE, 3)
    txt(mic_caption, mic_x, 140, WHITE, 13, anchor_x='center',
        anchor_y='center', bold=True)

    # ---- red SKIP: only after a wrong / unclear answer ----
    if g['voice_wrong'] and not g['voice_finished']:
        pixel_button(SKIP_X, MIC_Y, SKIP_W, SKIP_H, 'SKIP')

    draw_voice_bar()      # drawn last so it sits on top of everything


# WIN / GAME OVER panel is centered in the responsive logical viewport.
END_X, END_PLAY_Y, END_LEVELS_Y, END_BTN = W / 2, H / 2 - 33, H / 2 - 115, (300, 56)


def draw_end():
    draw_world()
    L, R = view['world_left'], view['world_right']
    B, T = view['world_bottom'], view['world_top']
    arcade.draw_lrbt_rectangle_filled(L, R, B, T, (20, 10, 40, 90))   # soft dim
    won = g['screen'] == 'win'
    hud_panel(END_X - 205, H / 2 - 165, 410, 330)

    if won:   # stars = hearts left
        for s in range(3):
            on = s < g['hearts']
            draw_star_shape(END_X - 44 + s * 44, H / 2 + 127, 16,
                            (255, 216, 97) if on else (88, 78, 120))
    boss_lvl = LEVELS[max(0, min(g['level'], NUM_LEVELS - 1))].get('boss', False)
    msg = ('BOSS DEFEATED!' if boss_lvl else 'YOU WIN!') if won else 'GAME OVER'
    pop = 1 + math.sin(g['t'] * 3) * 0.03
    hud_text(msg, END_X, H / 2 + (83 if won else 95), (255, 222, 133),
             round((40 if len(msg) <= 9 else 29) * pop), 'center')
    hud_text(f'Fruit collected: {min(g["score"], g["goal"])} / {g["goal"]}', END_X, H / 2 + 30,
             WHITE, 18, 'center')

    pixel_button(END_X, END_PLAY_Y, *END_BTN, 'PLAY AGAIN')
    pixel_button(END_X, END_LEVELS_Y, *END_BTN, 'LEVELS')   # back to the level page


def draw_game():
    window.clear()
    configure_viewport()
    with ui_camera.activate():
        if g['screen'] == 'start':
            draw_start()
        elif g['screen'] == 'characters':
            draw_character_select()
        elif g['screen'] == 'mode_select':
            draw_mode_select()
        elif g['screen'] == 'levels':
            draw_level_select()
        elif g['screen'] == 'voice':
            draw_voice()
        elif g['screen'] in ('break', 'break_done'):
            draw_break()
        elif g['screen'] in ('win', 'lose'):
            draw_end()
        else:
            draw_world()
        perf['d'] += 1
        if perf['show']:  # F3 toggles: updates/sec and frames/sec
            txt(f'UPS {perf["ups"]}  FPS {perf["fps"]}', W-10, 8, (83, 46, 113),
                12, anchor_x='right', bold=True)

# ---------------------------------------------------------------------------
# Camera + YOLO — runs in its own thread so the game never stutters.
# ---------------------------------------------------------------------------
_model_cache = {'model': None}


def hand_worker(stop_event, latest):
    """MediaPipe gestures from the SAME webcam frames as YOLO (no second camera)."""
    landmarker = None
    try:
        import mediapipe as mp
        from mediapipe.tasks import python as mp_python
        from mediapipe.tasks.python import vision
        if not HAND_MODEL_PATH.is_file():
            print('Hand model missing:', HAND_MODEL_PATH, flush=True)
            g['hand_status'] = 'MISSING MODEL'
            return
        options = vision.HandLandmarkerOptions(
            base_options=mp_python.BaseOptions(model_asset_path=str(HAND_MODEL_PATH)),
            running_mode=vision.RunningMode.VIDEO, num_hands=1)
        landmarker = vision.HandLandmarker.create_from_options(options)
        last_id, last_timestamp = -1, 0
        g['hand_status'] = 'READY'
        while not stop_event.is_set():
            current_id = latest['id']
            if current_id == last_id or latest['frame'] is None:
                time.sleep(0.006)
                continue
            last_id = current_id
            frame = cv2.flip(latest['frame'], 1)
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB,
                                data=np.ascontiguousarray(rgb))
            timestamp_ms = max(int(time.monotonic()*1000), last_timestamp+1)
            last_timestamp = timestamp_ms
            result = landmarker.detect_for_video(mp_image, timestamp_ms)
            gesture = 'STOP'
            if result.hand_landmarks:
                points = result.hand_landmarks[0]
                # Direction uses the hand, not facial features.
                # Same tested gesture mapping as the earlier working hand branch.
                dx = points[8].x - points[5].x
                dy = points[8].y - points[5].y
                if abs(dx) > abs(dy)*1.4:
                    gesture = 'RIGHT' if dx > 0 else 'LEFT'
                elif dy < -0.12:
                    gesture = 'UP'
            if stop_event.is_set():
                break
            g['hand_gesture'] = gesture
            g['hand_updated'] = time.monotonic()
    except Exception as err:
        print('MediaPipe hand tracking:', type(err).__name__, err, flush=True)
        g['hand_gesture'] = 'STOP'
        g['hand_status'] = 'HAND ERROR'
        g['hand_updated'] = 0.0
    finally:
        if landmarker is not None:
            landmarker.close()


def camera_worker(stop_event):
    cap = None
    try:
        if _model_cache['model'] is None:
            _model_cache['model'] = YOLO(str(ROOT/'best.pt'))
        model = _model_cache['model']
        try:  # warm-up so the first real frame isn't slow
            model.predict(np.zeros((YOLO_IMGSZ, YOLO_IMGSZ, 3), dtype=np.uint8),
                          imgsz=YOLO_IMGSZ, verbose=False)
        except Exception:
            pass
        if sys.platform.startswith('win'):
            cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
        else:
            cap = cv2.VideoCapture(0)
        if not cap.isOpened():
            cap.release()
            if not stop_event.is_set():
                g['cam_status'] = 'unavailable'
            return
        try:
            cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)  # newest frame, less lag
        except Exception:
            pass
    except Exception as err:
        print('Camera/YOLO:', err)
        if not stop_event.is_set():
            g['cam_status'] = 'unavailable'
        return

    if not stop_event.is_set():
        g['cam_status'] = 'ready'
    # A dedicated reader keeps only the NEWEST camera frame, so YOLO never
    # works on old buffered frames (this is what caused the delay).
    latest = {'frame': None, 'id': 0}
    done = threading.Event()

    def reader():
        while not done.is_set() and not stop_event.is_set():
            ok, f = cap.read()
            if ok:
                latest['frame'] = f
                latest['id'] += 1
            else:
                time.sleep(0.01)

    reader_thread = threading.Thread(target=reader, daemon=True)
    reader_thread.start()
    hand_thread = threading.Thread(
        target=hand_worker, args=(stop_event, latest), daemon=True)
    hand_thread.start()

    last, streak, last_id = None, 0, -1
    misses = 0
    try:
        while not stop_event.is_set():
            if latest['id'] == last_id:
                time.sleep(0.003)
                continue
            last_id = latest['id']
            frame = latest['frame']
            prediction = model.predict(frame, conf=YOLO_CONF, imgsz=YOLO_IMGSZ,
                                       verbose=False)[0]
            best_conf, detected = -1.0, None
            for box in prediction.boxes:
                label = prediction.names[int(box.cls.item())].lower()
                conf = float(box.conf.item())
                state = 'OPEN' if 'open' in label else (
                    'CLOSED' if 'clos' in label else None)
                if state is not None and conf > best_conf:
                    best_conf, detected = conf, state
            if stop_event.is_set():
                continue
            if detected is None:
                # MOUTH LOST: no mouth found = treat it as CLOSED (never stay OPEN)
                misses += 1
                if misses >= YOLO_LOST_FRAMES:
                    last, streak = 'CLOSED', 1
                    g['mouth'] = 'CLOSED'
                continue
            misses = 0
            # Debounce: switch only after the same reading repeats.
            if detected == last:
                streak += 1
            else:
                last, streak = detected, 1
            if streak >= YOLO_DEBOUNCE or g['mouth'] == 'OFF':
                g['mouth'] = detected
    except Exception as err:
        print('YOLO:', err)
        if not stop_event.is_set():
            g['cam_status'] = 'unavailable'
    finally:
        done.set()
        hand_thread.join(timeout=1.0)
        reader_thread.join(timeout=1.0)
        try:
            cap.release()
        except Exception:
            pass


def stop_camera():
    if g['cam_stop'] is not None:
        g['cam_stop'].set()
    th = g['cam_thread']
    if th is not None and th.is_alive() and th is not threading.current_thread():
        th.join(timeout=2.0)
    g['cam_stop'], g['cam_thread'] = None, None
    g['cam_status'], g['mouth'] = 'off', 'OFF'
    g['hand_gesture'], g['hand_updated'] = 'STOP', 0.0
    g['hand_status'] = 'WAITING'
    g['hand_jump_latched'] = False
    g['hand_jump_direction'] = 0


def start_camera():
    stop_camera()
    if not (ROOT/'best.pt').exists():
        g['cam_status'] = 'nomodel'
        return
    stop_event = threading.Event()
    th = threading.Thread(target=camera_worker, args=(stop_event,), daemon=True)
    g['cam_stop'], g['cam_thread'] = stop_event, th
    g['cam_status'] = 'loading'
    th.start()


# ---------------------------------------------------------------------------
# Game logic
# ---------------------------------------------------------------------------
def place_items(cfg, positions, items):
    """PLATFORMS: turn the item slots into (objects, platforms).

    One jump is ~133 high. Tier 1 platforms sit 76-92 above the ground and
    tier 2 sit 150-165 with a stepping stone beside them, so every platform
    is reachable. BOSS: `king` fruits (+3) are hidden on the highest ones."""
    objects, platforms = [], []
    plat, high = cfg.get('plat', 0.0), cfg.get('high', 0.0)
    for x, item in zip(positions, items):
        floor = GROUND
        if random.random() < plat:
            if random.random() < high:
                floor = GROUND + random.randint(150, 165)
                platforms.append({'x': float(x), 'y': float(floor), 'w': 130.0})
                platforms.append({'x': float(x) - 145,
                                  'y': float(GROUND + random.randint(76, 88)), 'w': 90.0})
            else:
                floor = GROUND + random.randint(76, 92)
                platforms.append({'x': float(x), 'y': float(floor), 'w': 130.0})
        objects.append({'kind': item, 'x': float(x), 'y': float(floor + 58),
                        'floor': floor, 'taken': False, 'pop': 0.0})
    king = cfg.get('king', 0)
    if king:
        fruits = [o for o in objects if o['kind'] in FRUIT_TYPES]
        top = [o for o in fruits if o['floor'] >= GROUND + 150]
        pool = sorted(top if len(top) >= king else fruits, key=lambda o: o['x'])
        picks = {(len(pool) * (i + 1)) // (king + 1) for i in range(king)}
        for i in picks:
            pool[i].update(kind='watermelon', king=True, bonus=3,
                           y=float(pool[i]['floor'] + 64))
    return objects, platforms


def new_game():
    """Start the level stored in g['level'] (set by start_level)."""
    cfg = LEVELS[g['level']]
    g.update(screen='playing', x=155.0, y=GROUND, vy=0.0, vx=0.0, cam=0.0,
             dcam=0, facing=1, keys=set(), hearts=3, score=0,
             goal=cfg['goal'],
             animation_time=0.0, invulnerable=0.0, eat_cooldown=0.0,
             coyote=0.0, jump_buffer=0.0, land=0.0, walk_phase=0.0,
             shake=0.0, particles=[], banner='', banner_time=0.0,
             heart_lost_t=0.0, heart_lost_i=0, fruit_pop=0.0,      # HUD
             hand_gesture='STOP', hand_updated=0.0, hand_status='WAITING',
             hand_jump_latched=False, hand_jump_direction=0,
             platforms=[], boss_intro=0.0,
             boss_hurt=0.0, boss_roar=0.0, boss_throw=1.2, boss_hazards=[])

    # Item slots along the level (same spacing logic as before).
    selected_positions = []
    x = random.randint(420, 500)
    while len(selected_positions) < 24 and x < WORLD_W - 150:
        while not (ground_exists(x) and ground_exists(x - 60)
                   and ground_exists(x + 60)):
            x += 20
        if x >= WORLD_W - 150:
            break
        selected_positions.append(x)
        x += random.randint(*ITEM_GAP)

    # Fruit = goal + 2 spare; bad items from the level table. Fruit always
    # gets priority so the goal is reachable even if few slots fit.
    fruit_count = min(cfg['goal'] + 2, len(selected_positions))
    bad_count = min(cfg['bad'], len(selected_positions) - fruit_count)
    bad_pool = BAD_ITEMS[:cfg['bad_n']]
    items = [random.choice(cfg['pool']) for _ in range(fruit_count)]
    items += [random.choice(bad_pool) for _ in range(bad_count)]
    random.shuffle(items)
    selected_positions = selected_positions[:len(items)]

    g['objects'], g['platforms'] = place_items(cfg, selected_positions, items)
    if cfg.get('boss'):                         # BOSS: warning intro + rumble
        g['boss_intro'], g['shake'] = 2.6, 0.3
    elif cfg.get('plat') and g['level'] == 2:   # first level with platforms: a hint
        set_banner('JUMP UP TO REACH HIGH FRUIT!')
    start_camera()


def start_level(index):
    """Open a level from the level page (only if it is unlocked)."""
    if 0 <= index < g['unlocked']:
        g['level'] = index
        new_game()


def finish_level():
    """Called when the goal is reached: stars = hearts left, unlock next."""
    lvl = g['level']
    g['stars'][lvl] = max(g['stars'][lvl], max(1, g['hearts']))
    g['unlocked'] = max(g['unlocked'], min(NUM_LEVELS, lvl + 2))
    save_progress()


def set_banner(message):
    g['banner'], g['banner_time'] = message, 1.35


def lose_heart(message, force=False):
    if g['invulnerable'] > 0 and not force:   # force: pits always cost a heart
        return
    g['hearts'] -= 1
    g['heart_lost_i'] = max(0, g['hearts'])   # HUD: which heart pops
    g['heart_lost_t'] = 0.5
    g['shake'] = 0.25
    play_effect('heart')
    g['invulnerable'] = 1.0
    set_banner(message)
    if g['hearts'] <= 0:
        g['screen'] = 'lose'
        g['vx'] = 0.0
        play_effect('game_over', 0.65)
        stop_camera()


def try_eat():
    if g['mouth'] != 'OPEN' or g['eat_cooldown'] > 0:
        return
    for thing in g['objects']:
        if thing['taken']:
            continue
        dx = thing['x'] - g['x']
        if abs(dx) < 73 and abs(thing['y']-(g['y']+55)) < 75 and dx*g['facing'] > -28:
            thing['taken'] = True
            thing['pop'] = 0.3
            g['eat_cooldown'] = 0.20
            if thing['kind'] in FRUIT_TYPES:
                bonus = thing.get('bonus', 1)     # BOSS: king fruit = +3
                g['score'] += bonus
                if is_boss_level():
                    boss_hit(bonus)               # BOSS: every fruit hurts him
                g['fruit_pop'] = 0.3          # HUD: fruit icon bounces
                play_effect('eat')
                if thing.get('king'):
                    set_banner('KING FRUIT! +3')
                    g['shake'] = 0.18
                    spawn_burst(thing['x'], thing['y'], (255, 214, 90), 26, speed=340)
                else:
                    set_banner('+1 FRUIT!')
                spawn_burst(thing['x'], thing['y'],
                            BURST_COLORS.get(thing['kind'], WHITE), 14)
                if g['score'] >= g['goal']:
                    finish_level()
                    if is_boss_level():           # BOSS: defeated -> explosion
                        bx, by = boss_screen_pos()
                        for c in ((255, 214, 90), (255, 96, 104), (120, 220, 110)):
                            spawn_burst(g['cam'] + bx, by, c, 22, speed=480,
                                        life=(0.6, 1.2))
                        g['boss_hazards'] = []
                    g['screen'] = 'win'
                    g['vx'] = 0.0  # stop leaning/sliding on the win screen
                    play_effect('win', 0.7)
                    stop_camera()
                    for _ in range(5):
                        spawn_burst(g['x'], g['y'] + 90, random.choice(CONFETTI),
                                    10, speed=430, size=(3, 6), grav=500.0,
                                    life=(0.8, 1.4))
            else:
                play_effect('wrong')
                spawn_burst(thing['x'], thing['y'], (150, 150, 165), 8, speed=180)
                lose_heart('NOT FOOD! -1 HEART')
            break


perf = {'last': None, 'u': 0, 'd': 0, 't0': time.perf_counter(),
        'ups': 0, 'fps': 0, 'show': False}


def update_game(_ignored_dt):
    """Drive the game from the REAL clock, not from the dt Arcade passes in.

    Whatever the refresh rate is (60/144 Hz) or however many times Arcade /
    Jupyter calls this per frame, game time always equals real time.
    """
    now = time.perf_counter()
    perf['u'] += 1
    if now - perf['t0'] >= 1.0:
        perf['ups'], perf['fps'] = perf['u'], perf['d']
        perf['u'] = perf['d'] = 0
        perf['t0'] = now
    last, perf['last'] = perf['last'], now
    if last is None:
        return
    step_game(min(now - last, 0.05))


def step_game(dt):
    raw_dt = dt
    dt = min(dt, 0.035)
    g['t'] += dt
    update_particles(dt)
    g['shake'] = max(0.0, g['shake'] - dt)
    g['heart_lost_t'] = max(0.0, g.get('heart_lost_t', 0.0) - dt)   # HUD
    g['fruit_pop'] = max(0.0, g.get('fruit_pop', 0.0) - dt)         # HUD
    g['voice_pop'] = max(0.0, g.get('voice_pop', 0.0) - dt)         # VOICE2
    for thing in g['objects']:
        if thing['pop'] > 0:
            thing['pop'] = max(0.0, thing['pop'] - dt)

    if g['screen'] == 'break':
        update_break(raw_dt)
        return
    if g['screen'] == 'voice':
        check_voice_result()
        if g['voice_wait'] > 0:
            g['voice_wait'] = max(0.0, g['voice_wait'] - raw_dt)
            if g['voice_wait'] == 0:
                next_voice_fruit()
        return
    if g['screen'] != 'playing':
        return

    g['animation_time'] += dt
    g['invulnerable'] = max(0.0, g['invulnerable']-dt)
    g['banner_time'] = max(0.0, g['banner_time']-dt)
    g['eat_cooldown'] = max(0.0, g['eat_cooldown']-dt)
    g['land'] = max(0.0, g['land'] - dt*6)
    g['boss_intro'] = max(0.0, g['boss_intro'] - dt)               # BOSS

    # --- HAND gestures: left/right motion, UP jumps ------------------------
    fresh = (g['hand_status'] == 'READY' and
             time.monotonic() - g['hand_updated'] < HAND_STALE_SECONDS)
    gesture = g['hand_gesture'] if fresh else 'STOP'
    direction = 1 if gesture == 'RIGHT' else (-1 if gesture == 'LEFT' else 0)
    # Keep walking momentum on UP, so jumping while moving is possible.
    if gesture == 'UP':
        if not g['hand_jump_latched']:
            g['hand_jump_direction'] = (1 if g['vx'] > 20 else
                                        -1 if g['vx'] < -20 else 0)
            g['jump_buffer'] = JUMP_BUFFER
            g['hand_jump_latched'] = True
        direction = g['hand_jump_direction']
    else:
        g['hand_jump_latched'] = False
        g['hand_jump_direction'] = 0
    airborne = standing_on() is None                # PLATFORMS: any surface counts
    if direction:
        g['facing'] = direction
        top = SPEED * (AIR_SPEED_MULT if airborne else 1.0)  # faster in the air
        rate = ACCEL * (2.0 if g['vx'] * direction < 0 else 1.0)  # snappy turns
        g['vx'] = approach(g['vx'], direction*top, rate*dt)
    else:
        fr = FRICTION * (AIR_FRICTION if airborne else 1.0)  # glide in the air
        g['vx'] = approach(g['vx'], 0.0, fr*dt)
    nx = g['x'] + g['vx']*dt
    if nx < 40 or nx > WORLD_W-60:
        nx = max(40, min(WORLD_W-60, nx))
        g['vx'] = 0.0
    if g['y'] < GROUND - 6 and not ground_exists(g['x']) and ground_exists(nx):
        for a, b in HOLES:                  # PIT FIX: inside a pit, the walls stop you
            if a <= g['x'] <= b:
                nx = max(a, min(b, nx))
                g['vx'] = 0.0
    g['x'] = nx

    # --- jumping: coyote time + input buffer + variable height --------------
    floor_now = standing_on()                       # PLATFORMS: ground or a platform top
    on_ground = floor_now is not None
    if on_ground:
        g['coyote'] = COYOTE
    else:
        g['coyote'] = max(0.0, g['coyote'] - dt)
    g['jump_buffer'] = max(0.0, g['jump_buffer'] - dt)
    if g['jump_buffer'] > 0 and g['coyote'] > 0:
        g['vy'] = JUMP_SPEED
        g['jump_buffer'] = g['coyote'] = 0.0
        play_effect('jump', 0.30)
        spawn_dust(g['x'], (floor_now if on_ground else GROUND) + 2, 6)

    # --- walk cycle follows the real speed ----------------------------------
    if on_ground and abs(g['vx']) > 10:
        before = int(g['walk_phase'])
        # CHAR_MOTION['anim']: faster frame cycle for flappy characters (pink).
        g['walk_phase'] += (abs(g['vx']) * dt / (SPEED / 8.0)) * char_motion()['anim']
        now = int(g['walk_phase'])
        if now != before and now % 2 == 0 and abs(g['vx']) > SPEED * 0.5 \
                and not char_motion()['hover']:   # flying monsters kick no dust
            spawn_dust(g['x'] - g['facing']*18, floor_now + 2, 2)

    prev_y = g['y']
    g['vy'] -= GRAVITY*dt
    g['y'] += g['vy']*dt
    land_on = floor_under(g['x'], prev_y)          # PLATFORMS: ground or platform top
    if land_on is not None and g['y'] <= land_on and g['vy'] < 0:
        if g['vy'] < -320:  # hard landing -> squash + dust
            g['land'] = min(1.0, -g['vy'] / 1100.0)
            spawn_dust(g['x'], land_on+2, 7)
        g['y'], g['vy'] = land_on, 0
    if g['y'] < -75:
        lose_heart('FELL IN A PIT! -1 HEART', force=True)
        if g['screen'] == 'playing':
            g['x'] = max(155, g['x']-300)
            while not ground_exists(g['x']):
                g['x'] -= 15
            g['y'], g['vy'], g['vx'] = GROUND, 0, 0.0
            g['coyote'] = 0.0

    # --- smooth camera with look-ahead --------------------------------------
    target = max(0, min(WORLD_W-W, g['x'] - W*0.38 + g['vx']*0.22))
    g['cam'] += (target - g['cam']) * (1 - math.exp(-CAM_SMOOTH*dt))
    g['cam'] = max(0, min(WORLD_W-W, g['cam']))
    try_eat()
    if g['screen'] == 'playing' and is_boss_level():
        update_boss(dt)                                   # BOSS


# ---------------------------------------------------------------------------
# Voice
# ---------------------------------------------------------------------------
def say_word_worker(word):
    if pyttsx3 is None:
        return 'pyttsx3 not installed. The word is: ' + word
    try:
        engine = pyttsx3.init()
        engine.say(word)
        engine.runAndWait()
        engine.stop()
        return 'Now say: ' + word
    except Exception as err:
        return 'Audio unavailable: ' + str(err)[:80]


def recognize_worker(stop_event):
    """Record PCM until the user taps the microphone again, then recognize it."""
    try:
        recognizer = sr.Recognizer()
        recognizer.dynamic_energy_threshold = True
        recognizer.pause_threshold = 0.8
        recognizer.operation_timeout = 10  # Avoid waiting forever on Google.

        with sr.Microphone() as source:
            recognizer.adjust_for_ambient_noise(source, duration=0.5)
            print('Microphone ready — speak, then tap the circle again.', flush=True)
            frames = []
            # Capture raw chunks until the stop button is pressed.
            while not stop_event.is_set():
                chunk = source.stream.read(source.CHUNK)
                frames.append(chunk)
            sample_rate = source.SAMPLE_RATE
            sample_width = source.SAMPLE_WIDTH

        if not frames:
            return ('error', 'No audio recorded. Try again.')

        audio = sr.AudioData(b''.join(frames), sample_rate, sample_width)
        print('Recording stopped. Recognizing...', flush=True)
        spoken = recognizer.recognize_google(audio, language='en-US')
        print('Google heard:', spoken, flush=True)
        return ('heard', spoken)

    except sr.UnknownValueError:
        return ('error', "Couldn't understand. Try again.")
    except (sr.RequestError, TimeoutError) as err:
        print('Google recognition error:', err, flush=True)
        return ('error', 'Speech service unavailable. Check internet.')
    except Exception as err:
        print('Microphone error:', err, flush=True)
        return ('error', 'Microphone error: ' + str(err)[:90])


def start_voice_task(kind):
    if g['voice_busy'] or g['voice_wait'] > 0 or g['voice_finished']:
        return
    stop_music()  # Keep microphone and spoken words clear.
    g['voice_busy'] = True
    g['voice_reaction'] = 'idle'
    g['voice_task_kind'] = kind
    if kind == 'recognize':
        stop_event = threading.Event()
        g['voice_stop_event'] = stop_event
        g['voice_recording'] = True
        g['voice_result'] = 'Recording... tap again to stop.'
        g['voice_future'] = voice_executor.submit(recognize_worker, stop_event)
    else:
        g['voice_result'] = 'Listen to the word...'
        g['voice_future'] = voice_executor.submit(
            say_word_worker, FRUIT_NAMES[g['voice_fruit']])


def stop_voice_recording():
    if not g['voice_recording']:
        return
    g['voice_recording'] = False
    g['voice_result'] = 'Checking your answer...'
    if g['voice_stop_event'] is not None:
        g['voice_stop_event'].set()


def toggle_voice_recording():
    if g['voice_recording']:
        stop_voice_recording()
    elif not g['voice_busy'] and g['voice_heard']:     # VOICE2: LISTEN comes first
        start_voice_task('recognize')


def next_voice_fruit():
    """VOICE2: move to the next fruit (after a correct answer or a SKIP)."""
    g['voice_heard'] = False
    g['voice_wrong'] = False
    if g['voice_index'] + 1 < len(g['voice_order']):
        g['voice_index'] += 1
        g['voice_fruit'] = g['voice_order'][g['voice_index']]
        g['voice_reaction'] = 'idle'
        g['voice_result'] = 'Press LISTEN to hear the word.'
    else:
        g['voice_finished'] = True
        g['voice_reaction'] = 'win'
        g['voice_result'] = f'You got {len(g["voice_got"])} / {len(FRUIT_TYPES)}!'
        play_effect('win', 0.7)


def skip_voice_fruit():
    """Skip the current fruit (only offered after a wrong answer); no point."""
    if (g['screen'] != 'voice' or g['voice_busy'] or g['voice_wait'] > 0
            or g['voice_finished'] or not g['voice_wrong']):
        return
    next_voice_fruit()
    if not g['voice_finished']:
        g['voice_result'] = 'Skipped! Try this fruit.'


def check_voice_result():
    future = g['voice_future']
    if future is None or not future.done():
        return
    g['voice_future'] = None
    g['voice_busy'] = False
    g['voice_recording'] = False
    g['voice_stop_event'] = None
    start_music()
    try:
        result = future.result()
    except Exception as err:
        g['voice_reaction'] = 'sad'
        g['voice_result'] = str(err)[:90]
        if g['voice_task_kind'] == 'speak':
            g['voice_heard'] = True            # never leave the player stuck
        return
    if g['voice_task_kind'] == 'speak':
        g['voice_result'] = result
        g['voice_heard'] = True                # VOICE2: the microphone unlocks now
        return
    state, spoken = result
    if state == 'error':
        g['voice_reaction'] = 'sad'
        g['voice_result'] = spoken
        g['voice_wrong'] = True                # VOICE2: SKIP appears
    elif FRUIT_NAMES[g['voice_fruit']].lower() in re.findall(r'[a-z]+', spoken.lower()):
        g['voice_score'] += 1
        g['voice_got'].add(g['voice_fruit'])   # VOICE2: fills the top bar
        g['voice_last'] = g['voice_fruit']
        g['voice_pop'] = 0.5
        g['voice_wrong'] = False
        g['voice_reaction'] = 'happy'
        play_effect('happy')
        g['voice_result'] = 'GREAT JOB! +1'
        g['voice_wait'] = VOICE_NEXT_DELAY
    else:
        g['voice_reaction'] = 'sad'
        g['voice_wrong'] = True                # VOICE2: SKIP appears
        play_effect('sad')
        g['voice_result'] = 'TRY AGAIN OR PRESS SKIP'



# ---------------------------------------------------------------------------
# BREAK THE FRUIT: automatic walking, colour then name, speech recognition.
# Colour choices correspond to the existing fruit artwork; edit if needed.
# ---------------------------------------------------------------------------
BREAK_COLORS = {'apple': ('red',), 'banana': ('yellow',),
    'strawberry': ('red',), 'orange': ('orange',), 'grapes': ('purple', 'green'),
    'watermelon': ('green', 'red'), 'mango': ('yellow', 'orange', 'green', 'red'),
    'kiwi': ('brown', 'green'), 'pineapple': ('yellow', 'green', 'brown'),
    'cherries': ('red',)}
BREAK_MIC = (388, 350, 200, 44)  # Above monster, below the question panel
BREAK_SKIP = (560, 350, 120, 44)
b = {}


def start_break():
    # Avoid overlapping microphone sessions when switching modes.
    if g['voice_future'] is not None and not g['voice_future'].done():
        return
    if b.get('future') is not None and not b['future'].done():
        return
    stop_camera()
    order = FRUIT_TYPES.copy()
    random.shuffle(order)
    b.clear()
    b.update(order=order, index=0, stage='walk', x=120.0, scroll=0.0,
             score=0, wrong=0, errors=0, started=time.monotonic(), elapsed=0.0,
             future=None, stop=None, recording=False, message='', effect=0.0,
             can_skip=False, skipped=0, cracked=False)
    g['screen'] = 'break'


def break_fruit():
    return b['order'][min(b['index'], len(b['order'])-1)]


def break_toggle_mic():
    if b['recording']:
        b['recording'] = False
        b['stop'].set()
        b['message'] = 'Checking your answer...'
        return
    if b['future'] is not None or b['stage'] not in ('color', 'name'):
        return
    stop_music()
    b['stop'] = threading.Event()
    b['recording'] = True
    b['message'] = 'Speak, then press STOP.'
    b['future'] = voice_executor.submit(recognize_worker, b['stop'])


def break_skip():
    """Skip only the current unanswered question; never award a point."""
    if (g['screen'] != 'break' or not b.get('can_skip', False)
            or b['future'] is not None or b['recording']
            or b['stage'] not in ('color', 'name')):
        return
    b['can_skip'] = False
    b['skipped'] += 1
    if b['stage'] == 'color':
        b.update(stage='name', message='Skipped! Now say the fruit name.')
    else:
        b.update(stage='skipped', effect=0.65, message='SKIPPED!')


def draw_break_skip():
    cx, cy, width, height = BREAK_SKIP
    enabled = b.get('can_skip', False) and b['future'] is None and not b['recording']
    if not enabled:
        pixel_button(*BREAK_SKIP, 'SKIP', enabled=False)
        return
    hover = hit(*g['mouse'], *BREAK_SKIP)
    left, bottom = cx-width/2, cy-height/2
    edge = (155, 40, 65)
    surface = (244, 100, 108) if hover else (220, 65, 80)
    pixel_rect(left+7, bottom-8, width-14, height, (70, 34, 100))
    pixel_rect(left, bottom+8, width, height-16, edge)
    pixel_rect(left+8, bottom, width-16, height, edge)
    pixel_rect(left+10, bottom+8, width-20, height-16, surface)
    pixel_rect(left+18, bottom+height-17, width-36, 6, (255, 155, 155))
    txt('SKIP', cx, cy, WHITE, 20, 'center', 'center', True)


def break_answer_matches(spoken, expected):
    # Match whole words rather than substrings; allow singular/plural forms.
    words = set(re.findall(r'[a-z]+', spoken.lower()))
    aliases = {'grapes': ('grape', 'grapes'), 'cherries': ('cherry', 'cherries'),
               'purple': ('purple', 'violet'), 'watermelon': ('watermelon', 'watermelons')}
    return any(word in words for answer in expected
               for word in aliases.get(answer, (answer, answer+'s')))


def update_break(dt):
    b['elapsed'] = time.monotonic() - b['started']
    if b['stage'] == 'walk':
        b['x'] = min(350.0, b['x'] + 175*dt)
        b['scroll'] += 95*dt
        if b['x'] >= 350:
            b['stage'] = 'color'
            b['message'] = 'What color is this fruit?'
    elif b['stage'] in ('shatter', 'skipped'):
        b['effect'] = max(0.0, b['effect'] - dt)
        if b['effect'] == 0:
            b['index'] += 1
            if b['index'] == len(b['order']):
                b['stage'] = 'done'
                g['screen'] = 'break_done'
                play_effect('win')
            else:
                b.update(stage='walk', x=120.0, message='', can_skip=False, cracked=False)
    f = b['future']
    if f is None or not f.done():
        return
    b.update(future=None, stop=None, recording=False)
    start_music()
    try:
        state, spoken = f.result()
    except Exception:
        state, spoken = 'error', 'Microphone unavailable. Try again.'
    if state == 'error':
        b['errors'] += 1
        if 'try again' in spoken.lower():
            b['can_skip'] = True
        b['message'] = spoken
        return
    fruit = break_fruit()
    expected = BREAK_COLORS[fruit] if b['stage'] == 'color' else (fruit,)
    if not break_answer_matches(spoken, expected):
        b['wrong'] += 1
        b['can_skip'] = True
        b['message'] = 'TRY AGAIN OR PRESS SKIP'
        play_effect('wrong')
        return
    b['can_skip'] = False
    b['score'] += 1
    play_effect('happy')
    if b['stage'] == 'color':
        b['stage'] = 'name'
        b['cracked'] = True
        b['message'] = 'Great! Now say the fruit name.'
    else:
        b.update(stage='shatter', effect=0.85, message='WELL DONE!')
        play_effect('eat')


def draw_break():
    draw_pixel_scene()
    hud_bar_body()
    draw_back_button(*hud_back_pos(), *HUD_BACK)
    hud_text(f'FRUIT {min(b["index"]+1, len(b["order"]))}/{len(b["order"])}',
             W/2, 568, WHITE, 18, 'center')
    secs = int(b['elapsed'])
    hud_text(f'{secs//60:02d}:{secs%60:02d}  |  {b["score"]}/{2*len(b["order"])}',
             W-20, 568, (255,222,133), 15, 'right')
    if b['stage'] == 'done':
        hud_panel(220, 135, 460, 340)
        hud_text('WELL DONE!', W/2, 425, (255,222,133), 34, 'center')
        hud_text(f'Correct answers: {b["score"]}/{2*len(b["order"])}', W/2, 365, WHITE, 19, 'center')
        hud_text(f'Wrong attempts: {b["wrong"]}  |  Skipped: {b["skipped"]}', W/2, 330, WHITE, 16, 'center')
        hud_text(f'Time: {secs//60:02d}:{secs%60:02d}', W/2, 295, WHITE, 19, 'center')
        pixel_button(W/2, 235, 290, 48, 'PLAY AGAIN')
        pixel_button(W/2, 172, 290, 48, 'GAMES')
        return
    stage = b['stage']
    pose = ('walk_'+str(int(g['t']*9)%4+1) if stage=='walk' else
            'happy' if stage=='shatter' else 'idle')
    bob = abs(math.sin(g['t']*10))*5 if stage=='walk' else 0
    w,h = MONSTER_GAME
    feet = sprite_bounds(current_character(), 'idle')[3]
    cy = 93 + bob - (1-feet)*h + h/2
    if current_character() == 'pink':
        cy += 25 + math.sin(g['t']*5)*5
    blit(character_tex(pose), b['x'], cy, w,h)
    fruit = break_fruit()
    # Full-resolution fruit texture, prepared lazily from existing PNG.
    key = (fruit, 'break', False)
    if key not in tex:
        tex[key] = make_tex(fruit, 240, 240)
    fx,fy = 650, 240
    if stage == 'shatter':
        progress = 1-b['effect']/0.85
        for i in range(16):
            a = i*math.tau/16
            r = 30+progress*150
            sz = max(2, 24*(1-progress))
            pixel_rect(fx+math.cos(a)*r, fy+math.sin(a)*r-80*progress**2,
                       sz,sz, (*BURST_COLORS[fruit], int(255*(1-progress))))
        blit(tex[key], fx, fy, 240*(1-progress),240*(1-progress),alpha=int(255*(1-progress)))
    else:
        blit(tex[key],fx,fy,240,240)
        if b['cracked'] and stage in ('name', 'skipped'):
            # Visible zig-zag crack after the correct colour answer.
            points = [(fx-15,fy+78),(fx+7,fy+38),(fx-13,fy+7),
                      (fx+18,fy-22),(fx-5,fy-64)]
            for a,c in zip(points,points[1:]):
                arcade.draw_line(*a,*c,(42,24,50),6)
                arcade.draw_line(a[0]+3,a[1],c[0]+3,c[1],(255,240,200),2)
    hud_panel(160,390,580,115)
    prompt = {'walk':'HERE COMES A FRUIT!', 'color':'WHAT COLOR IS IT?',
              'name':'WHAT FRUIT IS IT?', 'shatter':'FRUIT SMASHED!',
              'skipped':'NEXT FRUIT'}[stage]
    hud_text(prompt,W/2,465,(255,222,133),24,'center')
    msg = b['message']
    if len(msg)>65: msg=msg[:62]+'...'
    hud_text(msg or 'Get ready!', W/2,420,WHITE,15,'center')
    if stage in ('color','name'):
        label = 'STOP RECORDING' if b['recording'] else ('CHECKING...' if b['future'] else 'SPEAK')
        pixel_button(*BREAK_MIC,label,enabled=b['recording'] or b['future'] is None)
        draw_break_skip()


def leave_break():
    if b.get('stop') is not None:
        b['stop'].set()
    b['recording'] = False
    start_music()
    g['screen'] = 'mode_select'

# ---------------------------------------------------------------------------
# Input
# ---------------------------------------------------------------------------
def go_back():
    if g['screen'] in ('break', 'break_done'):
        leave_break()
        return
    if g['screen'] == 'voice':
        stop_voice_recording()
        start_music()
    if g['screen'] in ('playing', 'win', 'lose'):
        stop_camera()
    g['keys'].clear()
    # LEVELS: playing / win / lose go back to the level page.
    if g['screen'] in ('playing', 'win', 'lose'):
        g['screen'] = 'levels'
    elif g['screen'] in ('levels', 'voice'):
        g['screen'] = 'mode_select'
    elif g['screen'] == 'mode_select':
        g['screen'] = 'characters'
    elif g['screen'] == 'characters':
        g['screen'] = 'start'


def on_press(key, modifiers):
    # Optional F11 toggle; normal maximize works via the title bar.
    if key == arcade.key.F11:
        window.set_fullscreen(not window.fullscreen)
        configure_viewport()
        return
    if key == arcade.key.F3:
        perf['show'] = not perf['show']
        return
    if key == arcade.key.ESCAPE:
        go_back()
        return
    if g['screen'] in ('break', 'break_done'):
        if key == arcade.key.SPACE and g['screen'] == 'break':
            break_toggle_mic()
        elif key == arcade.key.ENTER and g['screen'] == 'break_done':
            start_break()
        return
    if g['screen'] == 'start':
        if key == arcade.key.ENTER:
            g['screen'] = 'characters'
        return
    if g['screen'] == 'characters':
        if key == arcade.key.ENTER:
            g['selected_character'] = current_character()
            g['screen'] = 'mode_select'
        return
    if g['screen'] == 'mode_select':
        if key == arcade.key.ENTER:
            g['screen'] = 'levels'
        return
    if g['screen'] == 'levels':
        # ENTER = newest unlocked level, keys 1-5 = that level.
        if key == arcade.key.ENTER:
            start_level(g['unlocked'] - 1)
        elif arcade.key.KEY_1 <= key <= arcade.key.KEY_1 + NUM_LEVELS - 1:
            start_level(key - arcade.key.KEY_1)
        return
    if g['screen'] == 'voice':
        if key == arcade.key.SPACE:
            toggle_voice_recording()
        elif key == arcade.key.ENTER:
            start_voice_task('speak')
        return
    if g['screen'] in ('win', 'lose'):
        if key == arcade.key.ENTER:
            new_game()
        return
    # C retries the shared camera; D toggles HAND and MOUTH status.
    if key == arcade.key.C:
        start_camera()
    elif key == arcade.key.D:
        g['show_debug'] = not g['show_debug']


def on_release(key, modifiers):
    # Walk and jump use the webcam hand gestures (not the keyboard).
    pass


def on_motion(x, y, dx, dy):
    g['mouse'] = screen_to_game(x, y)


def on_click(x, y, button, modifiers):
    x, y = screen_to_game(x, y)
    g['mouse'] = (x, y)
    if g['screen'] in ('break', 'break_done'):
        if hit(x,y,*hud_back_pos(),*HUD_BACK):
            leave_break()
        elif g['screen'] == 'break' and hit(x,y,*BREAK_MIC):
            break_toggle_mic()
        elif g['screen'] == 'break' and hit(x,y,*BREAK_SKIP):
            break_skip()
        elif g['screen'] == 'break_done':
            if hit(x,y,W/2,235,290,48): start_break()
            elif hit(x,y,W/2,172,290,48): leave_break()
        return
    # Clickable BACK on every screen after START; same action as Escape.
    # VOICE2: on the SAY THE FRUIT screen BACK lives in the top bar.
    top_back = g['screen'] in ('playing', 'voice')
    end_screen = g['screen'] in ('win', 'lose')
    back_pos = ((END_X, END_LEVELS_Y) if end_screen else
                hud_back_pos() if top_back else (90, 33))
    back_size = END_BTN if end_screen else (HUD_BACK if top_back else (138, 46))
    if g['screen'] in ('characters', 'mode_select', 'levels', 'voice',
                       'playing', 'win', 'lose'):
        if hit(x, y, *back_pos, *back_size):
            go_back()
            return
    if g['screen'] == 'start':
        if hit(x, y, W/2, 194, 225, 70):
            g['screen'] = 'characters'
    elif g['screen'] == 'characters':
        for index in range(len(CHARACTER_CHOICES)):
            cx = 150 + index * 200
            if hit(x, y, cx, 292, 172, 270) and character_ready(index):
                g['selected_character'] = CHARACTER_NAMES[index]
                g['screen'] = 'mode_select'
                break
    elif g['screen'] == 'mode_select':
        if hit(x, y, W/2, 379, 360, 70):
            g['screen'] = 'levels'      # LEVELS: open the level page
        elif hit(x, y, W/2, 175, 360, 70):
            if b.get('future') is not None and not b['future'].done():
                return
            if g['voice_future'] is not None and not g['voice_future'].done():
                return
            g.update(voice_future=None, voice_busy=False)
            voice_order = FRUIT_TYPES.copy()
            random.shuffle(voice_order)
            g.update(screen='voice', voice_score=0, voice_index=0,
                     voice_fruit=voice_order[0], voice_wait=0.0,
                     voice_finished=False, voice_order=voice_order,
                     voice_reaction='idle', voice_recording=False,
                     voice_stop_event=None,
                     voice_heard=False, voice_wrong=False,          # VOICE2
                     voice_got=set(), voice_last='', voice_pop=0.0,  # VOICE2
                     voice_result='Press LISTEN to hear the word.')
        elif hit(x, y, W/2, 277, 360, 70):
            start_break()
    elif g['screen'] == 'levels':
        for index in range(NUM_LEVELS):
            if hit(x, y, level_card_x(index), LV_CARD_Y, LV_CARD_W, LV_CARD_H):
                start_level(index)      # ignored automatically if locked
                break
    elif g['screen'] == 'voice':
        if hit(x, y, VP_X, VOICE_LISTEN_Y, 320, 54):
            start_voice_task('speak')
        elif (x - voice_mic_x())**2 + (y - MIC_Y)**2 <= MIC_R**2:
            toggle_voice_recording()
        elif g['voice_wrong'] and hit(x, y, SKIP_X, MIC_Y, SKIP_W, SKIP_H):
            skip_voice_fruit()
    elif g['screen'] in ('win', 'lose'):
        if hit(x, y, END_X, END_PLAY_Y, *END_BTN):
            new_game()


window.push_handlers(on_draw=draw_game, on_update=update_game,
                     on_key_press=on_press, on_key_release=on_release,
                     on_mouse_press=on_click, on_mouse_motion=on_motion,
                     on_resize=on_resize)
try:
    # Start as a normal resizable window. The user can maximize via the title-bar button.
    configure_viewport()
    start_music()
    arcade.run()
finally:
    if b.get('stop') is not None:
        b['stop'].set()
    if g['voice_stop_event'] is not None:
        g['voice_stop_event'].set()
    stop_music()
    stop_camera()
    voice_executor.shutdown(wait=False, cancel_futures=True)